# NB-04_salida_cg_coy

Process Flow SAS: **Salida CG coy** — `ProcessFlowContainer-8iloVuuXSe8CJiAb`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

Mes = None  # &Mes — nadie declaró su valor
Mes_imp1 = None  # &Mes_imp1 — nadie declaró su valor
Mes_imp10 = None  # &Mes_imp10 — nadie declaró su valor
Mes_imp11 = None  # &Mes_imp11 — nadie declaró su valor
Mes_imp12 = None  # &Mes_imp12 — nadie declaró su valor
Mes_imp2 = None  # &Mes_imp2 — nadie declaró su valor
Mes_imp3 = None  # &Mes_imp3 — nadie declaró su valor
Mes_imp4 = None  # &Mes_imp4 — nadie declaró su valor
Mes_imp5 = None  # &Mes_imp5 — nadie declaró su valor
Mes_imp6 = None  # &Mes_imp6 — nadie declaró su valor
Mes_imp7 = None  # &Mes_imp7 — nadie declaró su valor
Mes_imp8 = None  # &Mes_imp8 — nadie declaró su valor
Mes_imp9 = None  # &Mes_imp9 — nadie declaró su valor
Moneda = None  # &Moneda — nadie declaró su valor
PACAPR1 = None  # &PACAPR1 — nadie declaró su valor
PACAPR2 = None  # &PACAPR2 — nadie declaró su valor
PACAPR3 = None  # &PACAPR3 — nadie declaró su valor
PACAPR4 = None  # &PACAPR4 — nadie declaró su valor
PACAPR5 = None  # &PACAPR5 — nadie declaró su valor
PACAPR6 = None  # &PACAPR6 — nadie declaró su valor
PACAPR7 = None  # &PACAPR7 — nadie declaró su valor
Year = None  # &Year — nadie declaró su valor

faltantes = [n for n, v in {"Mes": Mes, "Mes_imp1": Mes_imp1, "Mes_imp10": Mes_imp10, "Mes_imp11": Mes_imp11, "Mes_imp12": Mes_imp12, "Mes_imp2": Mes_imp2, "Mes_imp3": Mes_imp3, "Mes_imp4": Mes_imp4, "Mes_imp5": Mes_imp5, "Mes_imp6": Mes_imp6, "Mes_imp7": Mes_imp7, "Mes_imp8": Mes_imp8, "Mes_imp9": Mes_imp9, "Moneda": Moneda, "PACAPR1": PACAPR1, "PACAPR2": PACAPR2, "PACAPR3": PACAPR3, "PACAPR4": PACAPR4, "PACAPR5": PACAPR5, "PACAPR6": PACAPR6, "PACAPR7": PACAPR7, "Year": Year}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text

# Conexión a BD — editable acá; SASMIG_DB_URL (orquestador) tiene
# prioridad si está definida (SUPUESTO: verificar servidor y base
# antes de correr contra datos reales).
config_db = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=PLATDAT,1433;"
    "DATABASE=GOBGENER;"
    "Authentication=ActiveDirectoryIntegrated;"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "MARS_Connection=Yes;"
)
engine = sqlalchemy.create_engine(
    os.environ.get("SASMIG_DB_URL", f"mssql+pyodbc:///?odbc_connect={config_db}"),
    pool_pre_ping=True,
    fast_executemany=True,
)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-04_salida_cg_coy.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## 1- TCons Mixtas_M_CGR

Redistribuye la muestra y el resto de transacciones mixtas (M) según la estructura de ponderadores, calcula el DEVENGO consolidado por SCN/agente/cuenta y produce las salidas totales (muestra + expansión) para reporte de Cuentas Nacionales, con ajuste puntual por imputación D63 en pandemia 2020

*confianza: medium · verificador: revise · SAS: PROC SQL CREATE TABLE con CASE-WHEN anidados sobre SUM, LEFT JOIN múltiples y UNION ALL*

In [ ]:
# ========= 1- TCons Mixtas_M_CGR =========
# CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MIXTAS (M) SEGUN ESTRUCTURA, PARA EJECUCION AÑO t
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m"))
sql_ejec_muestra_mixtas_m = f"""
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_CUP,
       (CASE WHEN t1.C_SCN='D11' THEN (SUM(t2.DEVENG/1000000*t1.POND))
             WHEN t1.C_SCN='P2' THEN (SUM(t2.DEVENG/1000000*t1.POND))
             WHEN t1.C_SCN='P51' THEN (SUM(t2.DEVENG/1000000*t1.POND))
             WHEN t1.C_SCN='D75' THEN (SUM(t2.DEVENG/1000000*t1.POND)) END) AS DEVENGO
INTO #ejec_muestra_mixtas_m
FROM GOBGENER.dbo.EST_MIXTAS_M t1
LEFT JOIN GOBGENER.dbo.EJECUCION t2
  ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
 AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA
 AND t2.C_ITEM = t1.C_ITEM AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24','33') AND t2.C_ITEM IN ('03','01','06','08','09')
  AND t2.AÑO = {int(Year)} AND t1.ENTIDAD NOT IN (5601,10201)
GROUP BY t2.MES,
         t2.AÑO,
         t1.C_SCN,
         t1.C_CUENTA,
         t1.C_ITEM,
         t1.ENTIDAD,
         t1.C_ASIGNACION,
         t1.C_ANALITICO,
         t1.C_CAGENTE,
         t1.C_CUP
"""
work_conn.execute(text(sql_ejec_muestra_mixtas_m))


In [ ]:
# DATA step: DROP de columnas de apoyo tras el cálculo agregado
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_drop"))
work_conn.execute(text(
    "SELECT * INTO #ejec_muestra_mixtas_m_drop FROM #ejec_muestra_mixtas_m"
))
# columnas PPTO_INICIAL, PPTO_ACTUAL, PERCIB_PAG, POR_PERCIB_PAG, DEVENG se descartan (DROP=)
work_conn.execute(text("ALTER TABLE #ejec_muestra_mixtas_m_drop DROP COLUMN PPTO_INICIAL, PPTO_ACTUAL, PERCIB_PAG, POR_PERCIB_PAG, DEVENG"))


In [ ]:
# CREATE TABLE ... SELECT DISTINCT T1.* -> EJEC_MUESTRA_MIXTAS_M final
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_final"))
work_conn.execute(text(
    "SELECT DISTINCT * INTO #ejec_muestra_mixtas_m_final FROM #ejec_muestra_mixtas_m_drop"
))
ejec_muestra_mixtas_m = pd.read_sql(text("SELECT * FROM #ejec_muestra_mixtas_m_final"), work_conn)
_log("ejec_muestra_mixtas_m", ejec_muestra_mixtas_m)


In [ ]:
# CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m"))
sql_cruce_ejec_exp_mixtas_m = f"""
SELECT t2.*,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.ENTIDAD,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN GOBGENER.dbo.EJECUCION t2
  ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
 AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA
 AND t2.C_ITEM = t1.C_ITEM
WHERE t2.C_CUENTA IN ('24','33') AND t2.C_ITEM IN ('03','01','06','08','09')
  AND t2.AÑO = {int(Year)}
  AND t1.M_MUESTRA = 0 AND t1.ENTIDAD NOT IN (5601,10201) AND t1.Clasif_Tcons = 'M'
GROUP BY t2.AÑO,
         t1.C_CUENTA,
         t1.C_ITEM,
         t1.ENTIDAD,
         t1.C_ASIGNACION,
         t1.C_ANALITICO
"""
work_conn.execute(text(sql_cruce_ejec_exp_mixtas_m))
cruce_ejec_exp_mixtas_m = pd.read_sql(text("SELECT * FROM #cruce_ejec_exp_mixtas_m"), work_conn)
_log("cruce_ejec_exp_mixtas_m", cruce_ejec_exp_mixtas_m)


In [ ]:
# CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M), PARA EJECUCION AÑO t
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim"))
sql_ejec_exp_mixtas_m_prelim = f"""
SELECT t2.*,
       t1.C_SCN,
       t1.C_CUP,
       t1.C_CAGENTE,
       SUM(t2.DEVENG) AS DATO_OR,
       (CASE WHEN t1.C_SCN='D11' THEN (SUM(t2.DEVENG/1000000*t1.POND*t3.DATO))
             WHEN t1.C_SCN='P2' THEN (SUM(t2.DEVENG/1000000*t1.POND*t3.DATO))
             WHEN t1.C_SCN='P51' THEN (SUM(t2.DEVENG/1000000*t1.POND*t3.DATO))
             WHEN t1.C_SCN='D75' THEN (SUM(t2.DEVENG/1000000*t1.POND*t3.DATO))
             WHEN t1.C_SCN='D73' THEN (SUM(t2.DEVENG/1000000*t1.POND*t3.DATO))
             WHEN t1.C_SCN='D63' THEN (SUM(t2.DEVENG/1000000*t1.POND*t3.DATO)) END) AS DEVENGO
INTO #ejec_exp_mixtas_m_prelim
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M t1
LEFT JOIN #cruce_ejec_exp_mixtas_m t2
  ON t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM AND t2.C_ASIGNACION = t1.C_ASIGNACION
 AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.MES = t1.MES AND t2.ENTIDAD = t1.ENTIDAD
LEFT JOIN #prom_mes t3
  ON t2.MONEDA = t3.MONEDA AND t2.MES = t3.MES AND t2.AÑO = t3.AÑO
WHERE t2.C_CUENTA IN ('24','33') AND t2.C_ITEM IN ('03','01','06','09','08') AND t2.AÑO = {int(Year)}
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.AÑO,
         t2.MES,
         t2.C_ENTIDAD,
         t2.C_PARTIDA,
         t2.N_PARTIDA,
         t2.C_CAPITULO,
         t2.N_CAPITULO,
         t2.C_PROGRAMA,
         t2.N_PROGRAMA,
         t2.C_TIPO_CUENTA,
         t2.C_CUENTA,
         t2.N_CUENTA,
         t2.C_ITEM,
         t2.C_ASIGNACION,
         t2.C_ANALITICO,
         t2.N_ANALITICO,
         t2.PPTO_INICIAL,
         t2.PPTO_ACTUAL,
         t2.DEVENG,
         t2.PERCIB_PAG,
         t2.POR_PERCIB_PAG,
         t2.ENTIDAD,
         t2.Clasif_Tcons, t1.C_SCN,
         t1.C_CAGENTE,
         t1.C_CUP
"""
work_conn.execute(text(sql_ejec_exp_mixtas_m_prelim))


In [ ]:
# DATA step: DROP de columnas de apoyo (PPTO_INICIAL PPTO_ACTUAL PERCIB_PAG POR_PERCIB_PAG DEVENG)
work_conn.execute(text("ALTER TABLE #ejec_exp_mixtas_m_prelim DROP COLUMN PPTO_INICIAL, PPTO_ACTUAL, PERCIB_PAG, POR_PERCIB_PAG, DEVENG"))
ejec_exp_mixtas_m_prelim = pd.read_sql(text("SELECT * FROM #ejec_exp_mixtas_m_prelim"), work_conn)
_log("ejec_exp_mixtas_m_prelim", ejec_exp_mixtas_m_prelim)


In [ ]:
# HOMOLOGACION TABLA DE EJECUCION DE RESTO DE TRANSACCIONES MIXTAS (M) EXPANDIDA A MUESTRA, ACORDE A TABLA EJECUCION MUESTRA
# El SELECT explícito de t1 NO incluye ENTIDAD ni Clasif_Tcons: se agregan como NULL
# para igualar el esquema de #ejec_muestra_mixtas_m_final y que el UNION ALL posterior
# empareje columnas por nombre y no por posición (hallazgo del verificador)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m"))
sql_ejec_exp_mixtas_m = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO, t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.N_PARTIDA,
       t1.C_CAPITULO,
       t1.N_CAPITULO,
       t1.C_PROGRAMA,
       t1.N_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.N_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       CAST(NULL AS int) AS ENTIDAD,
       CAST(NULL AS varchar(10)) AS Clasif_Tcons,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_CUP,
       t1.DEVENGO
INTO #ejec_exp_mixtas_m
FROM #ejec_exp_mixtas_m_prelim t1
"""
work_conn.execute(text(sql_ejec_exp_mixtas_m))


In [ ]:
# Crea tabla con la ejecucion total de transacciones Mixtas (M), Muestra + Resto
# UNION ALL por nombre explícito de columna (ambas fuentes ya comparten el mismo esquema)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total"))
cols_total = ("SUBSECTOR, MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO, "
              "N_CAPITULO, C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, "
              "C_ASIGNACION, C_ANALITICO, N_ANALITICO, ENTIDAD, Clasif_Tcons, C_SCN, C_CAGENTE, C_CUP, DEVENGO")
sql_ejec_mixtas_m_total = f"""
SELECT {cols_total} INTO #ejec_mixtas_m_total
FROM (
    SELECT {cols_total} FROM #ejec_muestra_mixtas_m_final
    UNION ALL
    SELECT {cols_total} FROM #ejec_exp_mixtas_m
) u
"""
work_conn.execute(text(sql_ejec_mixtas_m_total))


In [ ]:
# Ajuste por imputacion de D63 en pandemia (2020)
res = work_conn.execute(text(
    "DELETE FROM #ejec_mixtas_m_total "
    "WHERE C_ENTIDAD=911 AND C_CUENTA='24' AND C_ITEM='03' AND C_ASIGNACION='002' "
    "AND C_ANALITICO IN ('03002000') AND AÑO=2020 AND MES>4"
))
_log("DELETE #ejec_mixtas_m_total (ajuste D63 2020)", res.rowcount)
ejec_mixtas_m_total = pd.read_sql(text("SELECT * FROM #ejec_mixtas_m_total"), work_conn)
_log("ejec_mixtas_m_total", ejec_mixtas_m_total)


In [ ]:
# SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CG coyuntura
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt"))
sql_ejec_g_mixt = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t1.C_CUP,
       SUM(t1.DEVENGO) AS Dato, t3.ACTIVIDAD, t3.NIVEL
INTO #ejec_g_mixt
FROM #ejec_mixtas_m_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
LEFT JOIN GOBGENER.dbo.MOLDE_T_SCN_2 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
GROUP BY t1.SUBSECTOR,
         t1.MONEDA,
         t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MES,
         t1.C_TIPO_CUENTA,
         t1.C_CUENTA,
         t1.C_ITEM,
         t1.C_SCN,
         t1.C_CAGENTE, t3.ACTIVIDAD, t3.NIVEL,
         t2.C_TRANSACCION,
         t2.T_TRANSACCION,
         t2.N_CUENTA,
         t2.C_ACTIVO,
         t1.C_CUP
"""
work_conn.execute(text(sql_ejec_g_mixt))
ejec_g_mixt = pd.read_sql(text("SELECT * FROM #ejec_g_mixt"), work_conn)
_log("ejec_g_mixt", ejec_g_mixt)


In [ ]:
# SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para ANALISIS DESAGREGADO CG coyuntura
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_aper"))
sql_ejec_g_mixt_aper = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t1.C_CUP,
       SUM(t1.DEVENGO) AS Dato, t3.ACTIVIDAD, t3.NIVEL
INTO #ejec_g_mixt_aper
FROM #ejec_mixtas_m_total t1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t3 ON t1.C_ENTIDAD = t3.ENTIDAD
LEFT JOIN GOBGENER.dbo.MOLDE_T_SCN_2 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
GROUP BY t1.SUBSECTOR,
         t1.MONEDA,
         t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MES,
         t1.C_ENTIDAD,
         t1.C_PARTIDA,
         t1.C_CAPITULO,
         t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA,
         t1.C_CUENTA,
         t1.C_ITEM,
         t1.C_ASIGNACION,
         t1.C_ANALITICO,
         t1.N_ANALITICO,
         t1.C_SCN,
         t1.C_CAGENTE, t3.ACTIVIDAD, t3.NIVEL,
         t2.C_TRANSACCION,
         t2.T_TRANSACCION,
         t2.N_CUENTA,
         t2.C_ACTIVO,
         t1.C_CUP
"""
work_conn.execute(text(sql_ejec_g_mixt_aper))
ejec_g_mixt_aper = pd.read_sql(text("SELECT * FROM #ejec_g_mixt_aper"), work_conn)
_log("ejec_g_mixt_aper", ejec_g_mixt_aper)


In [ ]:
# DROP de las tablas WORK intermedias que ya cumplieron su función (equivalente al PROC SQL DROP TABLE final)
for t in ["#ejec_muestra_mixtas_m", "#ejec_muestra_mixtas_m_drop", "#cruce_ejec_exp_mixtas_m",
          "#ejec_exp_mixtas_m_prelim", "#ejec_exp_mixtas_m", "#ejec_mixtas_m_total"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## 5- TCons Mixtas_M_DIP_MOD

Redistribuye la ejecución presupuestaria DIPRES de transacciones mixtas (M) según la estructura de muestra por entidad, separando lo muestreado de lo expandido y consolidando la ejecución total para reporte de Cuentas Nacionales (CG coyuntura y análisis desagregado)

*confianza: medium · verificador: approve · SAS: PROC SQL: joins encadenados con CASE WHEN condicionado a C_SCN, agregaciones GROUP BY y UNION ALL de muestra + expansión*

In [ ]:
# ========= 5- TCons Mixtas_M_DIP_MOD =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
# Ejecucion dipres del periodo, con entidad asignada
# Year/Mes: macro vars documentadas (M-005) definidas en la celda parameters
sql_ejec_dip = f"""
SELECT T1.*, T2.C_ENTIDAD, T2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
    ON T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.AÑO = {int(Year)} AND T1.MES = {int(Mes)}
"""
work_conn.execute(text(sql_ejec_dip))
ejec_dip = pd.read_sql(text("SELECT * FROM #ejec_dip"), work_conn)
_log("ejec_dip", ejec_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_prev_dip"))
# TCONS MIXTAS DE LA MUESTRA DIPRES
# C_CUP FORMAT $CHAR4. LENGTH=4 -> se trunca/rellena a 4 caracteres explícitamente
sql_ejec_muestra_mixtas_m_prev_dip = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       CAST(t1.C_CUP AS char(4)) AS C_CUP,
       (CASE WHEN t1.C_SCN = 'D11' THEN SUM(t2.DEVENG / 1000000.0 * t1.POND)
             WHEN t1.C_SCN = 'P2'  THEN SUM(t2.DEVENG / 1000000.0 * t1.POND)
             WHEN t1.C_SCN = 'P51' THEN SUM(t2.DEVENG / 1000000.0 * t1.POND)
             WHEN t1.C_SCN = 'D75' THEN SUM(t2.DEVENG / 1000000.0 * t1.POND) END) AS DEVENGO
INTO #ejec_muestra_mixtas_m_prev_dip
FROM GOBGENER.dbo.EST_MIXTAS_M t1
LEFT JOIN #ejec_dip t2
    ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24', '33') AND t2.C_ITEM IN ('03', '01', '06', '08', '09')
  AND t1.ENTIDAD NOT IN (5601, 10201)
GROUP BY t2.MES, t2.AÑO, t1.C_SCN, t1.C_CUENTA, t1.C_ITEM, t1.ENTIDAD, t1.C_ASIGNACION,
         t1.C_ANALITICO, t1.C_CAGENTE, t1.C_CUP, t2.ACTIVIDAD, t2.NIVEL
"""
work_conn.execute(text(sql_ejec_muestra_mixtas_m_prev_dip))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_dip"))
# DATA step: descarta columnas de apoyo (DROP=)
sql_ejec_muestra_mixtas_m_dip = """
SELECT * INTO #ejec_muestra_mixtas_m_dip
FROM (
    SELECT t.*
    FROM #ejec_muestra_mixtas_m_prev_dip t
) src
"""
# se materializa sin PPTO_INICIAL, PPTO_ACTUAL, DEVENG, EJEC_ACUM
cols_res = pd.read_sql(text("SELECT TOP 0 * FROM #ejec_muestra_mixtas_m_prev_dip"), work_conn).columns.tolist()
cols_keep = [c for c in cols_res if c not in ("PPTO_INICIAL", "PPTO_ACTUAL", "DEVENG", "EJEC_ACUM")]
cols_keep_sql = ", ".join(cols_keep)
work_conn.execute(text(f"SELECT {cols_keep_sql} INTO #ejec_muestra_mixtas_m_dip FROM #ejec_muestra_mixtas_m_prev_dip"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_prev_dip"))
ejec_muestra_mixtas_m_dip = pd.read_sql(text("SELECT * FROM #ejec_muestra_mixtas_m_dip"), work_conn)
_log("ejec_muestra_mixtas_m_dip", ejec_muestra_mixtas_m_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m_dip"))
# CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t
sql_cruce_ejec_exp_mixtas_m_dip = """
SELECT t2.*,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.ENTIDAD,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m_dip
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_dip t2
    ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
WHERE t2.C_CUENTA IN ('24', '33') AND t2.C_ITEM IN ('03', '01', '06', '08', '09')
  AND t1.M_MUESTRA = 0 AND t1.ENTIDAD NOT IN (5601, 10201) AND t1.Clasif_Tcons = 'M'
GROUP BY t2.AÑO, t2.MES, t1.C_CUENTA, t1.C_ITEM, t1.ENTIDAD, t1.C_ASIGNACION, t1.C_ANALITICO
"""
work_conn.execute(text(sql_cruce_ejec_exp_mixtas_m_dip))
cruce_ejec_exp_mixtas_m_dip = pd.read_sql(text("SELECT * FROM #cruce_ejec_exp_mixtas_m_dip"), work_conn)
_log("cruce_ejec_exp_mixtas_m_dip", cruce_ejec_exp_mixtas_m_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #prom_mes"))
# PROM_MES es un WORK que dejó el nodo anterior en memoria (pandas); se sube para el cruce server-side
prom_mes.to_sql("#prom_mes", work_conn, if_exists="replace", index=False)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim_dip"))
# CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M)
# nota analista: JOINs en estructuras sin validación de unicidad -> puede duplicar filas si la estructura tiene más de un match
sql_ejec_exp_mixtas_m_prelim_dip = """
SELECT t2.*,
       t1.C_SCN,
       t1.C_CUP,
       t1.C_CAGENTE,
       SUM(t2.DEVENG) AS DATO_OR,
       (CASE WHEN t1.C_SCN = 'D11' THEN SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO)
             WHEN t1.C_SCN = 'P2'  THEN SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO)
             WHEN t1.C_SCN = 'P51' THEN SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO)
             WHEN t1.C_SCN = 'D75' THEN SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO)
             WHEN t1.C_SCN = 'D73' THEN SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO)
             WHEN t1.C_SCN = 'D63' THEN SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO) END) AS DEVENGO
INTO #ejec_exp_mixtas_m_prelim_dip
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M t1
LEFT JOIN #cruce_ejec_exp_mixtas_m_dip t2
    ON t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.MES = t1.MES AND t2.ENTIDAD = t1.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t2.MONEDA = t3.MONEDA AND t2.MES = t3.MES AND t2.AÑO = t3.AÑO
WHERE t2.C_CUENTA IN ('24', '33') AND t2.C_ITEM IN ('03', '01', '06', '09', '08')
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD, t2.C_PARTIDA, t2.C_CAPITULO,
         t2.C_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
         t2.N_ANALITICO, t2.PPTO_INICIAL, t2.PPTO_ACTUAL, t2.DEVENG, t2.ENTIDAD, t2.Clasif_Tcons,
         t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
work_conn.execute(text(sql_ejec_exp_mixtas_m_prelim_dip))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_dip"))
# DATA step: DROP de columnas de apoyo
cols_prelim = pd.read_sql(text("SELECT TOP 0 * FROM #ejec_exp_mixtas_m_prelim_dip"), work_conn).columns.tolist()
cols_drop_exp = ("PPTO_INICIAL", "PPTO_ACTUAL", "DEVENG", "EJEC_ACUM", "DATO_OR", "Clasif_Tcons", "ENTIDAD")
cols_keep_exp = [c for c in cols_prelim if c not in cols_drop_exp]
cols_keep_exp_sql = ", ".join(cols_keep_exp)
work_conn.execute(text(f"SELECT {cols_keep_exp_sql} INTO #ejec_exp_mixtas_m_dip FROM #ejec_exp_mixtas_m_prelim_dip"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim_dip"))
ejec_exp_mixtas_m_dip = pd.read_sql(text("SELECT * FROM #ejec_exp_mixtas_m_dip"), work_conn)
_log("ejec_exp_mixtas_m_dip", ejec_exp_mixtas_m_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total_dip"))
# Crea tabla con la ejecución total de transacciones Mixtas (M), Muestra + Resto
work_conn.execute(text("""
SELECT * INTO #ejec_mixtas_m_total_dip FROM #ejec_muestra_mixtas_m_dip
UNION ALL
SELECT * FROM #ejec_exp_mixtas_m_dip
"""))


In [ ]:
# SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CG coyuntura
# TRIM: función pura de MES (agrupado), se calcula sobre el agregado
sql_ejec_g_mixt_dip = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_CUP,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP AS C_CUP_1,
       SUM(t1.DEVENGO) AS Dato,
       t1.ACTIVIDAD,
       t1.NIVEL
INTO #ejec_g_mixt_dip
FROM #ejec_mixtas_m_total_dip t1
LEFT JOIN GOBGENER.dbo.MOLDE_T_SCN_2 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_SCN IN ('D11', 'P2', 'D63')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
         t1.ACTIVIDAD, t1.NIVEL, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP
"""
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_dip"))
work_conn.execute(text(sql_ejec_g_mixt_dip))
ejec_g_mixt_dip = pd.read_sql(text("SELECT * FROM #ejec_g_mixt_dip"), work_conn)
_log("ejec_g_mixt_dip", ejec_g_mixt_dip)


In [ ]:
# SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para ANALISIS DESAGREGADO CG coyuntura
sql_ejec_g_mixt_aper_dip = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_CUP,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP AS C_CUP_1,
       SUM(t1.DEVENGO) AS Dato,
       t1.ACTIVIDAD,
       t1.NIVEL
INTO #ejec_g_mixt_aper_dip
FROM #ejec_mixtas_m_total_dip t1
LEFT JOIN GOBGENER.dbo.MOLDE_T_SCN_2 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_SCN IN ('D11', 'P2', 'D63')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
         t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
         t1.ACTIVIDAD, t1.NIVEL, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP
"""
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_aper_dip"))
work_conn.execute(text(sql_ejec_g_mixt_aper_dip))
ejec_g_mixt_aper_dip = pd.read_sql(text("SELECT * FROM #ejec_g_mixt_aper_dip"), work_conn)
_log("ejec_g_mixt_aper_dip", ejec_g_mixt_aper_dip)


In [ ]:
# DROP TABLE final del SAS: limpieza de temporales de sesión (equivalente de scope)
for t in ["#ejec_muestra_mixtas_m_prev_dip", "#ejec_muestra_mixtas_m_dip", "#cruce_ejec_exp_mixtas_m_dip",
          "#ejec_exp_mixtas_m_prelim_dip", "#ejec_exp_mixtas_m_dip", "#ejec_mixtas_m_total_dip", "#ejec_dip"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


## 4- Salida EJEC hasta CGR

Ejecución presupuestaria de Gobierno Central hasta CGR: analíticos predefinidos y variables, transacciones diferenciadas por coyuntura/actividad, ajuste de transferencias en especie con imputaciones del DIPRES, consolidada en EJECUCION_CG_CGR

*confianza: medium · verificador: approve · SAS: PROC SQL encadenado: CREATE/UPDATE/APPEND sobre WORK con LEFT JOINs, CASE de mapeo de actividad, UNION ALL final*

In [ ]:
# ========= 4- Salida EJEC hasta CGR =========
# /*Analiticos predefinidos */
# CASE de C_ACT (mapeo ACTIVIDAD->AAPP/SSPP/EEPP) repetido en varias tablas
case_c_act = """CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
    WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
    WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
    WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
    WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
    WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
    WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
    WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
    WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END"""

work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim"))
sql_ejecucion_a_prelim = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       (CASE WHEN t2.ANALITICO='M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       ({case_c_act}) AS C_ACT,
       t4.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000.0) AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_prelim
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
  ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ENTIDAD NOT IN (5601,10201) AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejecucion_a_prelim))


In [ ]:
# /*Este Ajuste lo pidieron de síntesis, incorpora al indicador de transferencias en esp
#   lo que consideran desde anuales, el porcentaje sale de anuales...*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aj"))
sql_ejecucion_a_prelim_aj = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN,
       t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO,
       t1.C_CUP, t1.C_ACT, t1.NIVEL,
       (CASE WHEN C_CUENTA='22' AND C_ANALITICO='12999000' AND C_PARTIDA='16' AND C_ACT='SSPP'
             THEN t1.DEVENGO * 0.928 ELSE t1.DEVENGO END) AS DEVENGO
INTO #ejecucion_a_prelim_aj
FROM #ejecucion_a_prelim t1
"""
work_conn.execute(text(sql_ejecucion_a_prelim_aj))


In [ ]:
# UPDATE sobre EJECUCION_A_prelim_Aj: reasigna clasificación SCN de la fracción anual de transf. en especie
sql_update_aj = """
UPDATE #ejecucion_a_prelim_aj
SET C_SCN='D63', C_CAGENTE='S14', C_TRANSACCION='433', T_TRANSACCION='transfesp', N_CUENTA='IG', C_CUP='272'
WHERE C_CUENTA='22' AND C_ANALITICO='12999000' AND C_PARTIDA='16' AND C_ACT='SSPP'
"""
res = work_conn.execute(text(sql_update_aj))
_log("UPDATE #ejecucion_a_prelim_aj", res.rowcount)


In [ ]:
# Segundo CREATE TABLE EJECUCION_A_prelim: recalcula DEVENGO con el complemento (1-0.928)
# sobre la fila original (antes del ajuste), reemplazando la tabla completa
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_v2"))
sql_ejecucion_a_prelim_v2 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN,
       t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO,
       t1.C_CUP, t1.C_ACT, t1.NIVEL,
       (CASE WHEN C_CUENTA='22' AND C_ANALITICO='12999000' AND C_PARTIDA='16' AND C_ACT='SSPP'
             THEN t1.DEVENGO * (1 - 0.928) ELSE t1.DEVENGO END) AS DEVENGO
INTO #ejecucion_a_prelim_v2
FROM #ejecucion_a_prelim t1
"""
work_conn.execute(text(sql_ejecucion_a_prelim_v2))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim"))
work_conn.execute(text("SELECT * INTO #ejecucion_a_prelim FROM #ejecucion_a_prelim_v2"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_v2"))


In [ ]:
# Filtra de EJECUCION_A_prelim_Aj (ya con SCN/CAGENTE reasignados por el UPDATE) solo
# la fila de la fracción anual, para luego anexarla (APPEND) a EJECUCION_A_prelim
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aj_f"))
sql_ejecucion_a_prelim_aj_f = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN,
       t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO,
       t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO
INTO #ejecucion_a_prelim_aj_f
FROM #ejecucion_a_prelim_aj t1
WHERE t1.C_CUENTA='22' AND t1.C_ANALITICO='12999000' AND t1.C_PARTIDA='16' AND t1.C_ACT='SSPP'
"""
work_conn.execute(text(sql_ejecucion_a_prelim_aj_f))


In [ ]:
# PROC DATASETS APPEND BASE=EJECUCION_A_prelim DATA=EJECUCION_A_prelim_Aj FORCE
# -> INSERT server-side, columnas explícitas (mismo orden que EJECUCION_A_prelim)
cols_ejecucion_a_prelim = ("SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, "
                            "C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, C_ITEM, "
                            "C_ASIGNACION, C_ANALITICO, N_ANALITICO, ANALITICO_, C_SCN, "
                            "C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, "
                            "C_CUP, C_ACT, NIVEL, DEVENGO")
sql_append_prelim = f"""
INSERT INTO #ejecucion_a_prelim ({cols_ejecucion_a_prelim})
SELECT {cols_ejecucion_a_prelim} FROM #ejecucion_a_prelim_aj_f
"""
res = work_conn.execute(text(sql_append_prelim))
_log("APPEND #ejecucion_a_prelim", res.rowcount)


In [ ]:
# AUX_RATIOS: extracto de la fracción SSPP ya ajustada, insumo para los ratios de participación
work_conn.execute(text("DROP TABLE IF EXISTS #aux_ratios"))
sql_aux_ratios = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN,
       t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO,
       t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO
INTO #aux_ratios
FROM #ejecucion_a_prelim t1
WHERE t1.C_CUENTA='22' AND t1.C_PARTIDA='16' AND t1.C_ACT='SSPP'
"""
work_conn.execute(text(sql_aux_ratios))
aux_ratios = pd.read_sql(text("SELECT * FROM #aux_ratios"), work_conn)
_log("aux_ratios", aux_ratios)


In [ ]:
# TOTALES_INST: suma por institución (agregación pura de columnas del GROUP BY)
work_conn.execute(text("DROP TABLE IF EXISTS #totales_inst"))
sql_totales_inst = """
SELECT t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.C_ACT, t1.NIVEL, SUM(t1.DEVENGO) AS TOTAL_INST
INTO #totales_inst
FROM #aux_ratios t1
GROUP BY t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
         t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_SCN, t1.C_CAGENTE,
         t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_totales_inst))
totales_inst = pd.read_sql(text("SELECT * FROM #totales_inst"), work_conn)
_log("totales_inst", totales_inst)


In [ ]:
# TOTALES: suma por entidad/partida/capítulo/programa/cuenta (agregación pura del GROUP BY)
work_conn.execute(text("DROP TABLE IF EXISTS #totales"))
sql_totales = """
SELECT t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ACT, t1.NIVEL,
       SUM(t1.TOTAL_INST) AS TOTAL
INTO #totales
FROM #totales_inst t1
GROUP BY t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
         t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_totales))
totales = pd.read_sql(text("SELECT * FROM #totales"), work_conn)
_log("totales", totales)


In [ ]:
# RATIOS: participación de cada institución sobre el total de su entidad/año
work_conn.execute(text("DROP TABLE IF EXISTS #ratios"))
sql_ratios = """
SELECT t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.C_ACT, t1.NIVEL, t1.TOTAL_INST / t2.TOTAL AS Ratio
INTO #ratios
FROM #totales_inst t1
LEFT JOIN #totales t2 ON t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.AÑO = t2.AÑO
"""
work_conn.execute(text(sql_ratios))
ratios = pd.read_sql(text("SELECT * FROM #ratios"), work_conn)
_log("ratios", ratios)


In [ ]:
# /*Fin de ajuste....*/
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim WHERE C_CUENTA='32' AND C_ANALITICO IN ('04001000','04002000')"
))
_log("DELETE #ejecucion_a_prelim (32/04001000-04002000)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
#   que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim WHERE C_ENTIDAD=3801 AND C_CUENTA='06' AND C_ITEM='01' "
    "AND C_ASIGNACION='000' AND C_ANALITICO IN ('01000000')"
))
_log("DELETE #ejecucion_a_prelim (3801/06/01/000)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos (sigue)*/
# EJECUCION_A: agregación por analíticos predefinidos, sin desglose de entidad/partida/analítico
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a"))
sql_ejecucion_a = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a
FROM #ejecucion_a_prelim t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
         t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a))


In [ ]:
# /*Analiticos variables */
# EJECUCION_B: cuenta 32 con analíticos variables 04001000/04002000
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b"))
sql_ejecucion_b = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       ({case_c_act}) AS C_ACT, t4.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000.0) AS decimal(18,2)) AS Dato
INTO #ejecucion_b
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
     AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
  ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA='32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.C_ENTIDAD NOT IN (5601,10201) AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejecucion_b))


In [ ]:
# EJECUCION_C: cuentas 05/13/24/33 vía CR18_T_SCN_2, excluyendo obs de coyuntura/actividad
# y C_SCN='TC'. NOT IN de SAS descarta NULL (no se amplía a IS NULL OR ...)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c"))
sql_ejecucion_c = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       ({case_c_act}) AS C_ACT, t4.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000.0) AS decimal(18,2)) AS Dato
INTO #ejecucion_c
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
     AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
  ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejecucion_c))


In [ ]:
# EJECUCION_D: transacciones diferenciadas para coyuntura (OBS='CR18_difcoy'),
# C_ACTIVO pasa a ser C_SCN. NOT IN de SAS descarta NULL, sin ampliar con IS NULL
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d"))
sql_ejecucion_d = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t2.C_ACTIVO AS C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       ({case_c_act}) AS C_ACT, t4.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000.0) AS decimal(18,2)) AS Dato
INTO #ejecucion_d
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
     AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
  ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difcoy') AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejecucion_d))


In [ ]:
# EJECUCION_F: transacciones diferenciadas por actividad (OBS='CR18_difact'),
# C_CUP pasa a ser C_ACT. NOT IN de SAS descarta NULL, sin ampliar con IS NULL
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f"))
sql_ejecucion_f = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP, t2.C_CUP AS C_ACT, t4.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000.0) AS decimal(18,2)) AS Dato
INTO #ejecucion_f
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
     AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
  ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_SCN NOT IN ('TC')
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejecucion_f))


In [ ]:
# /* Transacciones TCONS (sólo aplica Mixtas) */
# EJEC_G_MIXT lo dejó CodeTask-RDZUBeSyggbaTi4d en memoria (pandas): se sube a #tmp
# para poder operar el resto de este bloque server-side, igual que las demás tablas
ejec_g_mixt.to_sql("#ejec_g_mixt", work_conn, if_exists="replace", index=False)


In [ ]:
# EJECUCION_G: agregación (función pura de columnas del GROUP BY) + mapeo C_ACT + DROP=ACTIVIDAD
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g"))
sql_ejecucion_g = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP,
       (CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL, t1.dato
INTO #ejecucion_g
FROM #ejec_g_mixt t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
         t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP,
         (CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
               WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
               WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
               WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
               WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
               WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
               WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
               WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
               WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END),
         t1.NIVEL, t1.dato
"""
work_conn.execute(text(sql_ejecucion_g))


In [ ]:
res = work_conn.execute(text("UPDATE #ejecucion_g SET C_ACTIVO='TC'"))
_log("UPDATE #ejecucion_g", res.rowcount)


In [ ]:
# /* Tabla con imputaciones filtrando por componentes CG coy */
# EJECUCION_CG_DIP_imputaciones lo produjo un nodo anterior en la BD/sesión (dependencia,
# no viene en el detalle 'en_memoria'): se lee de la sesión como #tmp existente
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imp"))
sql_ejecucion_cg_dip_imp = """
SELECT * INTO #ejecucion_cg_dip_imp
FROM #ejecucion_cg_dip_imputaciones
WHERE C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_ejecucion_cg_dip_imp))
# nota: DROP=KEY del SAS -> se excluye la columna KEY si existiera en el origen;
# como el SELECT * la arrastra, se elimina a continuación si está presente
work_conn.execute(text(
    "IF COL_LENGTH('tempdb..#ejecucion_cg_dip_imp', 'KEY') IS NOT NULL "
    "ALTER TABLE #ejecucion_cg_dip_imp DROP COLUMN KEY"
))


In [ ]:
# /*UNE TABLAS de Salida total de ejecución CG coy de data CGR*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr_prev"))
sql_ejecucion_cg_cgr_prev = """
SELECT * INTO #ejecucion_cg_cgr_prev FROM #ejecucion_a
UNION ALL
SELECT * FROM #ejecucion_b
UNION ALL
SELECT * FROM #ejecucion_c
UNION ALL
SELECT * FROM #ejecucion_d
UNION ALL
SELECT * FROM #ejecucion_g
UNION ALL
SELECT * FROM #ejecucion_f
UNION ALL
SELECT * FROM #ejecucion_cg_dip_imp
"""
work_conn.execute(text(sql_ejecucion_cg_cgr_prev))
ajecucion_cg_cgr_prev_check = pd.read_sql(text("SELECT COUNT(*) AS n FROM #ejecucion_cg_cgr_prev"), work_conn)
_log("ejecucion_cg_cgr_prev", ajecucion_cg_cgr_prev_check)


In [ ]:
for _t in ["#ejecucion_a_prelim", "#ejecucion_a", "#ejecucion_b", "#ejecucion_c", "#ejecucion_d", "#ejecucion_g", "#ejecucion_f", "#ejecucion_cg_dip_imp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# /*Incorporacion de imputaciones permanentes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imput_perm"))
sql_ejecucion_cg_dip_imput_perm = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP,
       (CASE WHEN C_ACT='FFAA' THEN 'AAPP'
             WHEN t1.C_ACT='FFOO' THEN 'AAPP'
             WHEN t1.C_ACT='PREVISION' THEN 'AAPP'
             WHEN t1.C_ACT='FONASA' THEN 'AAPP'
             WHEN t1.C_ACT='CORFO' THEN 'AAPP'
             WHEN t1.C_ACT='CENABAST' THEN 'AAPP'
             WHEN t1.C_ACT='AAPP' THEN 'AAPP'
             WHEN t1.C_ACT='SSPP' THEN 'SSPP'
             WHEN t1.C_ACT='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL, t1.Dato
INTO #ejecucion_cg_dip_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP t1
WHERE t1.AÑO = {int(Year)}
  AND t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_ejecucion_cg_dip_imput_perm))


In [ ]:
# EJECUCION_CG_CGR: salida final, une la agregación previa con las imputaciones permanentes
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr"))
sql_ejecucion_cg_cgr = """
SELECT * INTO #ejecucion_cg_cgr FROM #ejecucion_cg_cgr_prev
UNION ALL
SELECT * FROM #ejecucion_cg_dip_imput_perm
"""
work_conn.execute(text(sql_ejecucion_cg_cgr))
ejecucion_cg_cgr = pd.read_sql(text("SELECT * FROM #ejecucion_cg_cgr"), work_conn)
_log("ejecucion_cg_cgr", ejecucion_cg_cgr)


In [ ]:
for _t in ["#ejecucion_cg_dip_imput_perm", "#aux_ratios", "#totales_inst", "#totales"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## SALIDA CG_coy (CGR y DIP)

Ejecución de Gobierno Central de coyuntura (CGR+DIPRES) del año/mes de proceso, con apertura de analíticos predefinidos y variables, ajuste de transferencias en especie de salud y de ingresos operacionales, e incorporación de imputaciones permanentes

*confianza: medium · verificador: revise · SAS: Cadena de PROC SQL (LEFT JOIN, CASE, UNION ALL, GROUP BY) sobre tablas WORK y tablas de GOBGENER, con DELETE/UPDATE de depuración y APPEND FORCE*

In [ ]:
# ========= SALIDA CG_coy (CGR y DIP) =========
# /*Crea tabla de apertura por actividad por PACACR ante la falta de C_ENTIDAD en EJEC_DIPRES */
# /*MODIFICADO JUNIO 2025*/
# PROC SQL: Ejecucion dipres del periodo, con entidad asignada
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
sql_ejec_dip = f"""
SELECT T1.*, T2.C_ENTIDAD, T2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
    ON T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA
    AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.AÑO = {int(Year)} AND T1.MES = {int(Mes)}
"""
work_conn.execute(text(sql_ejec_dip))


In [ ]:
# /*Analiticos predefinidos*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip"))
sql_ejecucion_a_prelim_dip = """
SELECT t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
    t2.C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t2.C_ACTIVO,
    t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS DEVENGO,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_a_prelim_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip))


In [ ]:
# /*nuevo, para eliminar ingresos operacionales que vienen agregados en data dipres. Se incorporan al final del proceso*/
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE C_CUENTA = '07' AND C_ITEM = '0'"))
_log("DELETE #ejecucion_a_prelim_dip (item 07/0)", res.rowcount)


In [ ]:
# /*Estructura para abrir transferencias en especie...*/
work_conn.execute(text("DROP TABLE IF EXISTS #ratios2"))
sql_ratios2 = """
SELECT t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION,
    t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.Ratio
INTO #ratios2
FROM #ratios t1
WHERE t1.C_SCN = 'D63'
"""
work_conn.execute(text(sql_ratios2))


In [ ]:
# /*apertura transferencias en especie de la salud pública*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip2"))
sql_ejecucion_a_prelim_dip2 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION,
    t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO,
    t1.[KEY], t2.Ratio
INTO #ejecucion_a_prelim_dip2
FROM #ejecucion_a_prelim_dip t1
LEFT JOIN #ratios2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_CUENTA = t2.C_CUENTA AND t1.AÑO = t2.AÑO
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip2))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip"))
work_conn.execute(text("EXEC sp_rename '#ejecucion_a_prelim_dip2', '#ejecucion_a_prelim_dip'"))


In [ ]:
# EJECUCION_A_prelim_DIP_ajtesp: la apertura de transferencias en especie sale de
# los registros con Ratio no nulo (t1.Ratio not=.)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_ajtesp"))
sql_ejecucion_a_prelim_dip_ajtesp = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_,
    'D63' AS C_SCN, 'S14' AS C_CAGENTE, '433' AS C_TRANSACCION, 'transfesp' AS T_TRANSACCION,
    'IG' AS N_CUENTA, t1.C_ACTIVO, '272' AS C_CUP, t1.C_ACT, t1.NIVEL,
    t1.DEVENGO * t1.Ratio AS DEVENGO, t1.[KEY]
INTO #ejecucion_a_prelim_dip_ajtesp
FROM #ejecucion_a_prelim_dip t1
WHERE t1.Ratio IS NOT NULL
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip_ajtesp))


In [ ]:
# Recalcula DEVENGO: descuenta el ratio para los registros abiertos (case when Ratio not=.)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip3"))
sql_ejecucion_a_prelim_dip3 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION,
    t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
    (CASE WHEN t1.Ratio IS NOT NULL THEN t1.DEVENGO * (1 - t1.Ratio) ELSE t1.DEVENGO END) AS DEVENGO,
    t1.[KEY]
INTO #ejecucion_a_prelim_dip3
FROM #ejecucion_a_prelim_dip t1
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip3))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip"))
work_conn.execute(text("EXEC sp_rename '#ejecucion_a_prelim_dip3', '#ejecucion_a_prelim_dip'"))


In [ ]:
# PROC DATASETS APPEND FORCE: agrega la apertura (ajtesp) a la tabla base,
# alineando por nombre de columna
cols_ejecucion_a_prelim_dip = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, "
    "C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, ANALITICO_, "
    "C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, "
    "DEVENGO, [KEY]"
)
sql_append_ajtesp = f"""
INSERT INTO #ejecucion_a_prelim_dip ({cols_ejecucion_a_prelim_dip})
SELECT {cols_ejecucion_a_prelim_dip} FROM #ejecucion_a_prelim_dip_ajtesp
"""
res = work_conn.execute(text(sql_append_ajtesp))
_log("APPEND #ejecucion_a_prelim_dip (ajtesp)", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE [KEY] IN ('15_04_01')"))
_log("DELETE #ejecucion_a_prelim_dip (KEY 15_04_01)", res.rowcount)


In [ ]:
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"
))
_log("DELETE #ejecucion_a_prelim_dip (cuenta 32)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45*/
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip WHERE [KEY] IN ('07_06_01') AND C_CUENTA = '06' "
    "AND C_ITEM = '01' AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')"
))
_log("DELETE #ejecucion_a_prelim_dip (ajuste CORFO)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos (sigue)*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip"))
sql_ejecucion_a_dip = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
    SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a_dip
FROM #ejecucion_a_prelim_dip t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a_dip))
ejecucion_a_dip = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip"), work_conn)
_log("ejecucion_a_dip", ejecucion_a_dip)


In [ ]:
# /*Analiticos variables*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip"))
sql_ejecucion_b_dip = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t2.C_SCN, t2.C_CAGENTE,
    t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_b_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t1.C_ANALITICO IN ('04001000','04002000') AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_b_dip))
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip WHERE [KEY] IN ('15_04_01')"))
_log("DELETE #ejecucion_b_dip (KEY 15_04_01)", res.rowcount)
ejecucion_b_dip = pd.read_sql(text("SELECT * FROM #ejecucion_b_dip"), work_conn)
_log("ejecucion_b_dip", ejecucion_b_dip)


In [ ]:
# /*ANALITICOS VARIABLES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip"))
sql_ejecucion_c_dip = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t2.C_SCN, t2.C_CAGENTE,
    t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_c_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t2.C_SCN NOT IN ('TC') AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_c_dip))
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip WHERE [KEY] IN ('15_04_01')"))
_log("DELETE #ejecucion_c_dip (KEY 15_04_01)", res.rowcount)
ejecucion_c_dip = pd.read_sql(text("SELECT * FROM #ejecucion_c_dip"), work_conn)
_log("ejecucion_c_dip", ejecucion_c_dip)


In [ ]:
# /*Transacciones diferenciadas para coyuntura*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip"))
sql_ejecucion_d_dip = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t2.C_ACTIVO AS C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
    t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_d_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difcoy') AND t1.C_ENTIDAD NOT IN (5601,10201)
    AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t2.C_SCN NOT IN ('TC')
"""
work_conn.execute(text(sql_ejecucion_d_dip))
ejecucion_d_dip = pd.read_sql(text("SELECT * FROM #ejecucion_d_dip"), work_conn)
_log("ejecucion_d_dip", ejecucion_d_dip)


In [ ]:
# /*Transacciones diferenciadas por actividad*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip"))
sql_ejecucion_f_dip = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO,
    t2.C_CUP, t2.C_CUP AS C_ACT, t1.NIVEL,
    CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_f_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
    AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_ejecucion_f_dip))
ejecucion_f_dip = pd.read_sql(text("SELECT * FROM #ejecucion_f_dip"), work_conn)
_log("ejecucion_f_dip", ejecucion_f_dip)


In [ ]:
# EJECUCION_G_DIP: el GROUP BY del SAS NO incluye ACTIVIDAD ni C_ACT (C_ACT se
# calcula vía CASE sobre ACTIVIDAD dentro del SELECT pero queda fuera del GROUP BY);
# se agrupa exactamente por las columnas listadas en el SAS, sin ACTIVIDAD
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip"))
sql_ejecucion_g_dip = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP,
    (CASE WHEN MAX(t1.ACTIVIDAD) = 'FFAA' THEN 'AAPP'
          WHEN MAX(t1.ACTIVIDAD) = 'FFOO' THEN 'AAPP'
          WHEN MAX(t1.ACTIVIDAD) = 'PREVISION' THEN 'AAPP'
          WHEN MAX(t1.ACTIVIDAD) = 'FONASA' THEN 'AAPP'
          WHEN MAX(t1.ACTIVIDAD) = 'CORFO' THEN 'AAPP'
          WHEN MAX(t1.ACTIVIDAD) = 'CENABAST' THEN 'AAPP'
          WHEN MAX(t1.ACTIVIDAD) = 'AAPP' THEN 'AAPP'
          WHEN MAX(t1.ACTIVIDAD) = 'SSPP' THEN 'SSPP'
          WHEN MAX(t1.ACTIVIDAD) = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL, t1.dato,
    CONCAT_WS('_', t1.MONEDA) AS [KEY]
INTO #ejecucion_g_dip
FROM #ejec_g_mixt_dip t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP, t1.NIVEL, t1.dato
"""
work_conn.execute(text(sql_ejecucion_g_dip))


In [ ]:
res = work_conn.execute(text("UPDATE #ejecucion_g_dip SET C_ACTIVO = 'TC'"))
_log("UPDATE #ejecucion_g_dip (C_ACTIVO=TC)", res.rowcount)
ecjecucion_g_dip = pd.read_sql(text("SELECT * FROM #ejecucion_g_dip"), work_conn)
ejecucion_g_dip = ecjecucion_g_dip.drop(columns=["KEY"])
_log("ejecucion_g_dip", ejecucion_g_dip)


In [ ]:
# /*UNE TABLAS de Salida total de ejecución CG coy de data DIPRES*/
# UNION ALL de A,B,C,D,G,F (DROP=KEY); se replica excluyendo KEY de cada rama
cols_cg_dip = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, "
    "C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato"
)
sql_ejecucion_cg_dip = f"""
SELECT {cols_cg_dip} FROM #ejecucion_a_dip
UNION ALL
SELECT {cols_cg_dip} FROM #ejecucion_b_dip
UNION ALL
SELECT {cols_cg_dip} FROM #ejecucion_c_dip
UNION ALL
SELECT {cols_cg_dip} FROM #ejecucion_d_dip
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, "
    "C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, dato AS Dato FROM #ejecucion_g_dip
UNION ALL
SELECT {cols_cg_dip} FROM #ejecucion_f_dip
"""
ejecucion_cg_dip = pd.read_sql(text(sql_ejecucion_cg_dip), work_conn)
_log("ejecucion_cg_dip (union DIPRES)", ejecucion_cg_dip)


In [ ]:
# limpieza de temporales intermedias (equivalente al DROP TABLE del SAS)
for t in ["#ejecucion_a_prelim_dip", "#ejecucion_a_dip", "#ejecucion_b_dip", "#ejecucion_c_dip", "#ejecucion_d_dip", "#ejecucion_g_dip", "#ejecucion_f_dip"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
ejecucion_cg_dip["N_CUENTA"] = "DIPRES"
_log("ejecucion_cg_dip (N_CUENTA=DIPRES)", ejecucion_cg_dip)


In [ ]:
# /*Incorporacion de imputaciones permanentes*/
sql_ejecucion_cg_dip_imput_perm = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP,
    (CASE WHEN t1.C_ACT = 'FFAA' THEN 'AAPP'
          WHEN t1.C_ACT = 'FFOO' THEN 'AAPP'
          WHEN t1.C_ACT = 'PREVISION' THEN 'AAPP'
          WHEN t1.C_ACT = 'FONASA' THEN 'AAPP'
          WHEN t1.C_ACT = 'CORFO' THEN 'AAPP'
          WHEN t1.C_ACT = 'CENABAST' THEN 'AAPP'
          WHEN t1.C_ACT = 'AAPP' THEN 'AAPP'
          WHEN t1.C_ACT = 'SSPP' THEN 'SSPP'
          WHEN t1.C_ACT = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL, t1.Dato
FROM GOBGENER.dbo.IMPUT_GC_DIP t1
WHERE t1.AÑO = {int(Year)} AND t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
ejecucion_cg_dip_imput_perm = pd.read_sql(text(sql_ejecucion_cg_dip_imput_perm), engine)
_log("ejecucion_cg_dip_imput_perm", ejecucion_cg_dip_imput_perm)


In [ ]:
ejecucion_cg_dip = pd.concat([ejecucion_cg_dip, ejecucion_cg_dip_imput_perm], ignore_index=True)
_log("ejecucion_cg_dip (con imput_perm)", ejecucion_cg_dip)


In [ ]:
# /*NUEVO: IMPUTA INGRESOS OPERACIONALES PARA LAS ENTIDADES QUE LO INFORMAN AGREGADO EN DIPRES, ASÍ SE MEJORA EL INDICADOR DE VENTAS*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_07"))
sql_ejecucion_a_dip_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
    t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO,
    t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_dip_07
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('07') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_ejecucion_a_dip_07))
ejecucion_a_dip_07 = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip_07"), work_conn)
_log("ejecucion_a_dip_07", ejecucion_a_dip_07)


In [ ]:
# /*Verifica que entidades informan solo el agregado para filtrarlas*/
work_conn.execute(text("DROP TABLE IF EXISTS #cuenta_item_07"))
sql_cuenta_item_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP,
    COUNT(t1.DEVENGO) AS id
INTO #cuenta_item_07
FROM #ejecucion_a_dip_07 t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP
"""
work_conn.execute(text(sql_cuenta_item_07))
cuenta_item_07 = pd.read_sql(text("SELECT * FROM #cuenta_item_07"), work_conn)
_log("cuenta_item_07", cuenta_item_07)


In [ ]:
sql_ejecucion_dip_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
    'DIPRES' AS N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
    t1.devengo AS Dato
FROM #ejecucion_a_dip_07 t1
LEFT JOIN #cuenta_item_07 t2
    ON t1.MONEDA = t2.MONEDA AND t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.C_PARTIDA = t2.C_PARTIDA
    AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
WHERE t2.id = 1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.C_ACTIVO,
    t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.devengo
"""
ejecucion_dip_07 = pd.read_sql(text(sql_ejecucion_dip_07), work_conn)
_log("ejecucion_dip_07", ejecucion_dip_07)


In [ ]:
ejecucion_cg_dip = pd.concat([ejecucion_cg_dip, ejecucion_dip_07], ignore_index=True)
ejecucion_cg_dip = ejecucion_cg_dip[ejecucion_cg_dip["nivel"].fillna("") != "EXCLUIDO"] if "nivel" in ejecucion_cg_dip.columns else ejecucion_cg_dip[ejecucion_cg_dip["NIVEL"].fillna("") != "EXCLUIDO"]
_log("ejecucion_cg_dip (final DIPRES, sin EXCLUIDO)", ejecucion_cg_dip)


In [ ]:
for t in ["#ejecucion_a_dip_07", "#cuenta_item_07", "#ejec_dip", "#ejecucion_a_prelim_dip_ajtesp", "#ratios2"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
# /*UNE TABLAS de Salida total de ejecución CG coy con info completa, CGR + DIPRES (mes t)*/
ejecucion_cg = pd.concat([ejecucion_cg_cgr, ejecucion_cg_dip], ignore_index=True)
_log("ejecucion_cg", ejecucion_cg)


## Aper_CG_coy hasta CGR

Ejecución presupuestaria de Gobierno Central desagregada por apertura institucional (AAPP/SSPP/EEPP), analíticos predefinidos y variables, con ajustes de coyuntura para transferencias especiales, Beca Especial Carbón y consolidación hasta CGR incorporando imputaciones permanentes de DIPRES

*confianza: medium · verificador: unverified · SAS: PROC SQL: múltiples CREATE TABLE con CASE-WHEN anidados, joins con tablas de clasificación (CR18_T_SCN_1/2, T_SCN_2, CR18_APERTURA), UPDATE puntuales, UNION ALL y APPEND FORCE*

In [ ]:
# ========= Aper_CG_coy hasta CGR =========
# Analiticos predefinidos: EJECUCION_A_prelim_aper — CASE-WHEN de TRIM (trimestre) y de C_ACT (agrupación institucional), join con SCN_1, PROM_MES y CR18_APERTURA
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aper"))
sql_ejecucion_a_prelim_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN(1,2,3) THEN 1
             WHEN t1.MES IN(4,5,6) THEN 2
             WHEN t1.MES IN(7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       (SUM(t1.DEVENG*t3.DATO/1000000)) AS DEVENGO
INTO #ejecucion_a_prelim_aper
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2 ON (t1.C_TIPO_CUENTA = t2.T_CUENTA) AND (t1.C_CUENTA = t2.C_CUENTA)
    AND (t1.C_ANALITICO = t2.C_ANALITICO)
LEFT JOIN #prom_mes t3 ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4 ON (t1.C_ENTIDAD = t4.ENTIDAD)
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ENTIDAD NOT IN (5601,10201) AND t1.AÑO = {int(Year)}
GROUP BY
       t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN(1,2,3) THEN 1
             WHEN t1.MES IN(4,5,6) THEN 2
             WHEN t1.MES IN(7,8,9) THEN 3
             ELSE 4 END),
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END),
       t4.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a_prelim_aper))


In [ ]:
# /*Este Ajuste lo pidieron de síntesis, incorpora al indicador de transferencias en esp lo que consideran desde anuales, el porcentaje sale de anuales...*/
# Split 92.8% / 7.2% del devengo de la cuenta 22 (SSPP) con reclasificación SCN a D63/transfesp
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_a_prelim_aper_aj"))
sql_ejecucion_a_prelim_aper_aj = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.C_ACT, t1.NIVEL,
       (CASE WHEN t1.C_CUENTA='22' AND t1.C_ANALITICO='12999000' AND t1.C_PARTIDA='16' AND t1.C_ACT='SSPP' THEN t1.DEVENGO*0.928 ELSE t1.DEVENGO END) AS DEVENGO
INTO #ejec_a_prelim_aper_aj
FROM #ejecucion_a_prelim_aper t1
"""
work_conn.execute(text(sql_ejecucion_a_prelim_aper_aj))

# UPDATE: reclasifica el 92.8% como transferencia especial (D63/transfesp) para la porción ajustada
sql_update_aj = """
UPDATE #ejec_a_prelim_aper_aj
SET C_SCN='D63', C_CAGENTE='S14', C_TRANSACCION='433', T_TRANSACCION='transfesp', N_CUENTA='IG', C_CUP='272'
WHERE C_CUENTA='22' AND C_ANALITICO='12999000' AND C_PARTIDA='16' AND C_ACT='SSPP'
"""
res = work_conn.execute(text(sql_update_aj))
_log("UPDATE #ejec_a_prelim_aper_aj (reclasificación D63)", res.rowcount)


In [ ]:
# Recalcula EJECUCION_A_prelim_aper con el 7.2% restante (complemento del ajuste anterior, sin reclasificar)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aper_v2"))
sql_ejecucion_a_prelim_aper_v2 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.C_ACT, t1.NIVEL,
       (CASE WHEN t1.C_CUENTA='22' AND t1.C_ANALITICO='12999000' AND t1.C_PARTIDA='16' AND t1.C_ACT='SSPP' THEN t1.DEVENGO*(1-0.928) ELSE t1.DEVENGO END) AS DEVENGO
INTO #ejecucion_a_prelim_aper_v2
FROM #ejecucion_a_prelim_aper t1
"""
work_conn.execute(text(sql_ejecucion_a_prelim_aper_v2))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aper"))
work_conn.execute(text("SELECT * INTO #ejecucion_a_prelim_aper FROM #ejecucion_a_prelim_aper_v2"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aper_v2"))


In [ ]:
# EJECUCION_A_prelim_aper_Aj final: solo las filas del ajuste (con el 92.8% ya reclasificado a D63/transfesp)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_a_prelim_aper_aj_final"))
sql_aj_final = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.C_ACT, t1.NIVEL, t1.DEVENGO
INTO #ejec_a_prelim_aper_aj_final
FROM #ejec_a_prelim_aper_aj t1
WHERE t1.C_CUENTA='22' AND t1.C_ANALITICO='12999000' AND t1.C_PARTIDA='16' AND t1.C_ACT='SSPP'
"""
work_conn.execute(text(sql_aj_final))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_a_prelim_aper_aj"))
work_conn.execute(text("SELECT * INTO #ejec_a_prelim_aper_aj FROM #ejec_a_prelim_aper_aj_final"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_a_prelim_aper_aj_final"))

# PROC DATASETS APPEND FORCE: agrega la porción reclasificada (D63) al resto
cols_ejec_a_prelim_aper = "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, DEVENGO"
sql_append_aj = f"""
INSERT INTO #ejecucion_a_prelim_aper ({cols_ejec_a_prelim_aper})
SELECT {cols_ejec_a_prelim_aper} FROM #ejec_a_prelim_aper_aj
"""
res = work_conn.execute(text(sql_append_aj))
_log("APPEND #ejecucion_a_prelim_aper (ajuste transfesp)", res.rowcount)
# /*Fin de ajuste....*/


In [ ]:
# DELETE: elimina analíticos de la cuenta 32 predefinidos (se tratan aparte en EJECUCION_B)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_aper WHERE C_CUENTA='32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_aper (cuenta 32)", res.rowcount)

# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_aper WHERE C_ENTIDAD=3801 AND C_CUENTA='06' AND C_ITEM='01' AND C_ASIGNACION='000' AND C_ANALITICO IN ('01000000')"))
_log("DELETE #ejecucion_a_prelim_aper (CORFO)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos (sigue)*/ EJECUCION_A_aper: agregación final por dimensión completa
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_aper"))
sql_ejecucion_a_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       t1.C_ACT, t1.NIVEL, (SUM(t1.DEVENGO)) AS Dato
INTO #ejecucion_a_aper
FROM #ejecucion_a_prelim_aper t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
         t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
         t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a_aper))
ejecucion_a_prelim_aper = pd.read_sql(text("SELECT * FROM #ejecucion_a_prelim_aper"), work_conn)
_log("ejecucion_a_prelim_aper", ejecucion_a_prelim_aper)
ajecucion_a_prelim_aper_aj = pd.read_sql(text("SELECT * FROM #ejec_a_prelim_aper_aj"), work_conn)
ejecucion_a_prelim_aper_aj = ajecucion_a_prelim_aper_aj
_log("ejecucion_a_prelim_aper_aj", ejecucion_a_prelim_aper_aj)


In [ ]:
# /*Analiticos variables*/ EJECUCION_B_aper: cuenta 32 (analíticos 04001000/04002000) contra T_SCN_2
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_aper"))
sql_ejecucion_b_aper = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN(1,2,3) THEN 1
             WHEN t1.MES IN(4,5,6) THEN 2
             WHEN t1.MES IN(7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG*t3.DATO/1000000) AS Dato
INTO #ejecucion_b_aper
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.T_SCN_2 t2 ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA AND
           t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION =
           t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO)
LEFT JOIN #prom_mes t3 ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4 ON (t1.C_ENTIDAD = t4.ENTIDAD)
WHERE t1.C_CUENTA='32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.C_ENTIDAD NOT IN (5601,10201) AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejecucion_b_aper))
ejecucion_b_aper = pd.read_sql(text("SELECT * FROM #ejecucion_b_aper"), work_conn)
_log("ejecucion_b_aper", ejecucion_b_aper)


In [ ]:
# EJECUCION_C_aper: cuentas 05/13/24/33 contra CR18_T_SCN_2 (join incluye C_ENTIDAD)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_aper"))
sql_ejecucion_c_aper = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN(1,2,3) THEN 1
             WHEN t1.MES IN(4,5,6) THEN 2
             WHEN t1.MES IN(7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG*t3.DATO/1000000) AS Dato
INTO #ejecucion_c_aper
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
           AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
           AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO)
LEFT JOIN #prom_mes t3 ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4 ON (t1.C_ENTIDAD = t4.ENTIDAD)
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND
      t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC') AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejecucion_c_aper))

# /* Beca Especial Carbon (previo 2020) */ tres UPDATE puntuales sobre el mismo registro
for _col, _val in [("C_CAGENTE", "'S14'"), ("C_SCN", "'D75'"), ("C_TRANSACCION", "'4345'")]:
    res = work_conn.execute(text(
        f"UPDATE #ejecucion_c_aper SET {_col}={_val} "
        "WHERE C_ENTIDAD=6403 AND C_CUENTA='24' AND C_ITEM='01' AND C_ASIGNACION='270' "
        "AND C_ANALITICO IN ('01270000') AND MONEDA IN ('P') AND AÑO<2020"
    ))
    _log(f"UPDATE #ejecucion_c_aper ({_col})", res.rowcount)
ejecucion_c_aper = pd.read_sql(text("SELECT * FROM #ejecucion_c_aper"), work_conn)
_log("ejecucion_c_aper", ejecucion_c_aper)


In [ ]:
# /* Transacciones diferenciadas para coyuntura */ EJECUCION_D_aper: OBS='CR18_difcoy', C_SCN toma el valor de C_ACTIVO
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_aper"))
sql_ejecucion_d_aper = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN(1,2,3) THEN 1
             WHEN t1.MES IN(4,5,6) THEN 2
             WHEN t1.MES IN(7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_ACTIVO AS C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG*t3.DATO/1000000) AS Dato
INTO #ejecucion_d_aper
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
           AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
           AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO)
LEFT JOIN #prom_mes t3 ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4 ON (t1.C_ENTIDAD = t4.ENTIDAD)
WHERE t2.OBS IN ('CR18_difcoy') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.C_SCN NOT IN ('TC') AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejecucion_d_aper))
ejecucion_d_aper = pd.read_sql(text("SELECT * FROM #ejecucion_d_aper"), work_conn)
_log("ejecucion_d_aper", ejecucion_d_aper)


In [ ]:
# /* Transacciones diferenciadas por actividad */ EJECUCION_F_aper: OBS='CR18_difact', join SIN C_ENTIDAD, C_ACT=C_CUP
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_aper"))
sql_ejecucion_f_aper = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN(1,2,3) THEN 1
             WHEN t1.MES IN(4,5,6) THEN 2
             WHEN t1.MES IN(7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
       t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
       t2.C_ACTIVO, t2.C_CUP,
       t2.C_CUP AS C_ACT, t4.NIVEL,
       (t1.DEVENG*t3.DATO/1000000) AS Dato
INTO #ejecucion_f_aper
FROM GOBGENER.dbo.EJECUCION t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
           AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
           AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO)
LEFT JOIN #prom_mes t3 ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4 ON (t1.C_ENTIDAD = t4.ENTIDAD)
WHERE t2.OBS IN ('CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejecucion_f_aper))
ejecucion_f_aper = pd.read_sql(text("SELECT * FROM #ejecucion_f_aper"), work_conn)
_log("ejecucion_f_aper", ejecucion_f_aper)


In [ ]:
# EJECUCION_G_aper: agrupa EJEC_G_MIXT_APER (dejado en memoria por la dependencia 1- TCons Mixtas_M_CGR) recalculando C_ACT desde ACTIVIDAD
ewg_mixt_aper = ejec_g_mixt_aper
ewg_mixt_aper.to_sql("#ejec_g_mixt_aper", work_conn, if_exists="replace", index=False)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_aper"))
sql_ejecucion_g_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       (CASE WHEN ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL, t1.dato
INTO #ejecucion_g_aper
FROM #ejec_g_mixt_aper t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
         t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
         t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         (CASE WHEN ACTIVIDAD='FFAA' THEN 'AAPP'
               WHEN ACTIVIDAD='FFOO' THEN 'AAPP'
               WHEN ACTIVIDAD='PREVISION' THEN 'AAPP'
               WHEN ACTIVIDAD='FONASA' THEN 'AAPP'
               WHEN ACTIVIDAD='CORFO' THEN 'AAPP'
               WHEN ACTIVIDAD='CENABAST' THEN 'AAPP'
               WHEN ACTIVIDAD='AAPP' THEN 'AAPP'
               WHEN ACTIVIDAD='SSPP' THEN 'SSPP'
               WHEN ACTIVIDAD='EEPP' THEN 'EEPP' END),
         t1.NIVEL, t1.dato
"""
work_conn.execute(text(sql_ejecucion_g_aper))
ewg_mixt_aper_g = pd.read_sql(text("SELECT * FROM #ejecucion_g_aper"), work_conn)
ejecucion_g_aper = ewg_mixt_aper_g
_log("ejecucion_g_aper", ejecucion_g_aper)


In [ ]:
# /*UNE TABLAS*/ EJECUCION_CG_CGR_aper_prev: UNION ALL de las 7 fuentes (incluye EJECUCION_CG_DIP_aper_imp, dejada en memoria por SALIDA CG_coy CGR y DIP)
# nombre real en memoria: la dependencia dejó WORK.EJECUCION_CG_DIP_IMP -> se sube como #ejecucion_cg_dip_aper_imp
ejecucion_cg_dip_imp.to_sql("#ejecucion_cg_dip_aper_imp", work_conn, if_exists="replace", index=False)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr_aper_prev"))
sql_union = """
SELECT * INTO #ejecucion_cg_cgr_aper_prev FROM #ejecucion_a_aper
UNION ALL
SELECT * FROM #ejecucion_b_aper
UNION ALL
SELECT * FROM #ejecucion_c_aper
UNION ALL
SELECT * FROM #ejecucion_d_aper
UNION ALL
SELECT * FROM #ejecucion_g_aper
UNION ALL
SELECT * FROM #ejecucion_f_aper
UNION ALL
SELECT * FROM #ejecucion_cg_dip_aper_imp
"""
work_conn.execute(text(sql_union))
ejecucion_cg_cgr_aper_prev = pd.read_sql(text("SELECT * FROM #ejecucion_cg_cgr_aper_prev"), work_conn)
_log("ejecucion_cg_cgr_aper_prev", ejecucion_cg_cgr_aper_prev)

# limpieza de tablas intermedias (equivalente al DROP TABLE del SAS; EJECUCION_CG_DIP_aper_imp queda comentado/preservado en el original)
for _t in ["#ejecucion_a_prelim_aper", "#ejecucion_a_aper", "#ejecucion_b_aper", "#ejecucion_c_aper", "#ejecucion_d_aper", "#ejecucion_g_aper", "#ejecucion_f_aper"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# /*Incorporacion de imputaciones permanentes*/ EJEC_CG_DIP_aper_IMPUT_PERM desde tabla de BD IMPUT_GC_DIP_APER
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cg_dip_aper_imput_perm"))
sql_ejec_cg_dip_aper_imput_perm = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
       t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
       t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
       t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       (CASE WHEN t1.C_ACT='FFAA' THEN 'AAPP'
             WHEN t1.C_ACT='FFOO' THEN 'AAPP'
             WHEN t1.C_ACT='PREVISION' THEN 'AAPP'
             WHEN t1.C_ACT='FONASA' THEN 'AAPP'
             WHEN t1.C_ACT='CORFO' THEN 'AAPP'
             WHEN t1.C_ACT='CENABAST' THEN 'AAPP'
             WHEN t1.C_ACT='AAPP' THEN 'AAPP'
             WHEN t1.C_ACT='SSPP' THEN 'SSPP'
             WHEN t1.C_ACT='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL, t1.Dato
INTO #ejec_cg_dip_aper_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER t1
WHERE t1.AÑO = {int(Year)}
  AND t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_ejec_cg_dip_aper_imput_perm))
ejec_cg_dip_aper_imput_perm = pd.read_sql(text("SELECT * FROM #ejec_cg_dip_aper_imput_perm"), work_conn)
_log("ejec_cg_dip_aper_imput_perm", ejec_cg_dip_aper_imput_perm)


In [ ]:
# EJECUCION_CG_CGR_aper final: une el consolidado previo con las imputaciones permanentes
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr_aper"))
sql_final = """
SELECT * INTO #ejecucion_cg_cgr_aper FROM #ejecucion_cg_cgr_aper_prev
UNION ALL
SELECT * FROM #ejec_cg_dip_aper_imput_perm
"""
work_conn.execute(text(sql_final))
ejecucion_cg_cgr_aper = pd.read_sql(text("SELECT * FROM #ejecucion_cg_cgr_aper"), work_conn)
_log("ejecucion_cg_cgr_aper", ejecucion_cg_cgr_aper)

# limpieza final (equivalente al DROP TABLE del SAS)
for _t in ["#ejec_cg_dip_aper_imput_perm", "#ejec_a_prelim_aper_aj"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## APER_CG_coy (CGR y DIP)

Ejecución presupuestaria del gobierno central de la fuente DIPRES para el periodo, clasificada por transacción SCN y actividad (analíticos predefinidos, variables, diferenciados por coyuntura y por actividad), consolidada con la salida CGR

*confianza: medium · verificador: approve · SAS: PROC SQL: LEFT JOINs sucesivos, CASE WHEN, GROUP BY, DELETE/UPDATE, UNION ALL de analíticos DIPRES y consolidación con CGR*

In [ ]:
# ========= APER_CG_coy (CGR y DIP) =========
# /*MODIFICADO JUNIO 2025*/ /*Ejecucion dipres del periodo, con entidad asignada*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
sql_ejec_dip = f"""
SELECT T1.*, T2.C_ENTIDAD, T2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
  ON T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA
 AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.AÑO = {int(Year)} AND T1.MES = {int(Mes)}
"""
work_conn.execute(text(sql_ejec_dip))
ejec_dip = pd.read_sql(text("SELECT * FROM #ejec_dip"), work_conn)
_log("ejec_dip", ejec_dip)


In [ ]:
# /*Analiticos predefinidos*/
# CASE de C_ACT sobre ACTIVIDAD, replicado igual en A/B/C/D/F/G
caso_c_act = """
    CASE WHEN T1.ACTIVIDAD='FFAA' THEN 'AAPP'
         WHEN T1.ACTIVIDAD='FFOO' THEN 'AAPP'
         WHEN T1.ACTIVIDAD='PREVISION' THEN 'AAPP'
         WHEN T1.ACTIVIDAD='FONASA' THEN 'AAPP'
         WHEN T1.ACTIVIDAD='CORFO' THEN 'AAPP'
         WHEN T1.ACTIVIDAD='CENABAST' THEN 'AAPP'
         WHEN T1.ACTIVIDAD='AAPP' THEN 'AAPP'
         WHEN T1.ACTIVIDAD='SSPP' THEN 'SSPP'
         WHEN T1.ACTIVIDAD='EEPP' THEN 'EEPP' END
"""
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
sql_ejecucion_a_prelim_dip_aper = f"""
SELECT T1.SUBSECTOR, T1.MONEDA, T1.AÑO,
    (CASE WHEN T1.MES IN (1,2,3) THEN 1
          WHEN T1.MES IN (4,5,6) THEN 2
          WHEN T1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    T1.MES, T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA,
    T1.C_TIPO_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION,
    T1.C_ANALITICO, T1.N_ANALITICO,
    (CASE WHEN T2.ANALITICO='M' THEN T1.N_ANALITICO ELSE T2.ANALITICO END) AS ANALITICO_,
    T2.C_SCN, T2.C_CAGENTE, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA,
    T2.C_ACTIVO, T2.C_CUP,
    {caso_c_act} AS C_ACT,
    T1.NIVEL,
    CAST((T1.DEVENG * T3.DATO / 1000000) AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper
FROM #ejec_dip T1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 T2
  ON T1.C_TIPO_CUENTA = T2.T_CUENTA AND T1.C_CUENTA = T2.C_CUENTA AND T1.C_ANALITICO = T2.C_ANALITICO
LEFT JOIN #prom_mes T3
  ON T1.MONEDA = T3.MONEDA AND T1.MES = T3.MES AND T1.AÑO = T3.AÑO
WHERE T1.C_CUENTA NOT IN ('05','13','24','33') AND T2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND T1.C_ENTIDAD NOT IN (5601,10201) AND T1.AÑO = {int(Year)} AND T1.MES = {int(Mes)}
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip_aper))


In [ ]:
# DELETE FROM EJECUCION_A_prelim_DIP_aper WHERE C_ENTIDAD IN (10201)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_a_prelim_dip_aper (10201)", res.rowcount)


In [ ]:
# DELETE analiticos 32 / 04001000-04002000
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA='32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_dip_aper (32)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper "
    "WHERE C_ENTIDAD IN (3801) AND C_CUENTA='06' AND C_ITEM='01' AND C_ASIGNACION='000' AND C_ANALITICO IN ('01000000')"
))
_log("DELETE #ejecucion_a_prelim_dip_aper (CORFO)", res.rowcount)
ejecucion_a_prelim_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_a_prelim_dip_aper"), work_conn)
_log("ejecucion_a_prelim_dip_aper", ejecucion_a_prelim_dip_aper)


In [ ]:
# /*Analiticos predefinidos (sigue)*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_aper"))
sql_ejecucion_a_dip_aper = """
SELECT T1.SUBSECTOR, T1.MONEDA, T1.AÑO, T1.TRIM, T1.MES, T1.C_ENTIDAD,
    T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA, T1.C_CUENTA,
    T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO, T1.C_SCN,
    T1.C_CAGENTE, T1.C_TRANSACCION, T1.T_TRANSACCION, T1.N_CUENTA, T1.C_ACTIVO,
    T1.C_CUP, T1.C_ACT, T1.NIVEL,
    T1.DEVENGO AS Dato
INTO #ejecucion_a_dip_aper
FROM #ejecucion_a_prelim_dip_aper T1
GROUP BY T1.SUBSECTOR, T1.MONEDA, T1.AÑO, T1.TRIM, T1.MES, T1.C_ENTIDAD,
    T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA, T1.C_CUENTA,
    T1.C_ITEM, T1.C_SCN, T1.C_CAGENTE, T1.C_TRANSACCION, T1.T_TRANSACCION,
    T1.N_CUENTA, T1.C_ACTIVO, T1.C_CUP, T1.C_ACT, T1.NIVEL,
    T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO, T1.DEVENGO
"""
# nota: T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO no estaban en el GROUP BY
# original de SAS (PROC SQL toma el valor del primer registro del grupo); se
# agregan al GROUP BY porque T-SQL exige toda columna no agregada ahí
work_conn.execute(text(sql_ejecucion_a_dip_aper))
ejecucion_a_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip_aper"), work_conn)
_log("ejecucion_a_dip_aper", ejecucion_a_dip_aper)


In [ ]:
# /*Analiticos variables*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip_aper"))
sql_ejecucion_b_dip_aper = f"""
SELECT T1.SUBSECTOR, T1.MONEDA, T1.AÑO,
    (CASE WHEN T1.MES IN (1,2,3) THEN 1
          WHEN T1.MES IN (4,5,6) THEN 2
          WHEN T1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    T1.MES, T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA,
    T1.C_TIPO_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION,
    T1.C_ANALITICO, T1.N_ANALITICO,
    T2.C_SCN, T2.C_CAGENTE, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA,
    T2.C_ACTIVO, T2.C_CUP,
    {caso_c_act} AS C_ACT,
    T1.NIVEL,
    CAST((T1.DEVENG * T3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_b_dip_aper
FROM #ejec_dip T1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 T2
  ON T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
 AND T1.C_TIPO_CUENTA = T2.T_CUENTA AND T1.C_CUENTA = T2.C_CUENTA AND T1.C_ITEM = T2.C_ITEM
 AND T1.C_ASIGNACION = T2.C_ASIGNACION AND T1.C_ANALITICO = T2.C_ANALITICO AND T1.C_ENTIDAD = T2.ENTIDAD
LEFT JOIN #prom_mes T3
  ON T1.MONEDA = T3.MONEDA AND T1.MES = T3.MES AND T1.AÑO = T3.AÑO
WHERE T1.C_CUENTA='32' AND T2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND T1.C_ANALITICO IN ('04001000','04002000')
  AND T1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_b_dip_aper))
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_b_dip_aper (10201)", res.rowcount)
ejecucion_b_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_b_dip_aper"), work_conn)
_log("ejecucion_b_dip_aper", ejecucion_b_dip_aper)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_aper"))
sql_ejecucion_c_dip_aper = f"""
SELECT T1.SUBSECTOR, T1.MONEDA, T1.AÑO,
    (CASE WHEN T1.MES IN (1,2,3) THEN 1
          WHEN T1.MES IN (4,5,6) THEN 2
          WHEN T1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    T1.MES, T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA,
    T1.C_TIPO_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION,
    T1.C_ANALITICO, T1.N_ANALITICO,
    T2.C_SCN, T2.C_CAGENTE, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA,
    T2.C_ACTIVO, T2.C_CUP,
    {caso_c_act} AS C_ACT,
    T1.NIVEL,
    CAST((T1.DEVENG * T3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_c_dip_aper
FROM #ejec_dip T1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 T2
  ON T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
 AND T1.C_TIPO_CUENTA = T2.T_CUENTA AND T1.C_CUENTA = T2.C_CUENTA AND T1.C_ITEM = T2.C_ITEM
 AND T1.C_ASIGNACION = T2.C_ASIGNACION AND T1.C_ANALITICO = T2.C_ANALITICO AND T1.C_ENTIDAD = T2.ENTIDAD
LEFT JOIN #prom_mes T3
  ON T1.MONEDA = T3.MONEDA AND T1.MES = T3.MES AND T1.AÑO = T3.AÑO
WHERE T1.C_CUENTA IN ('05','13','24','33') AND T2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND T2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND T1.C_ENTIDAD NOT IN (5601,10201) AND T2.C_SCN NOT IN ('TC')
"""
work_conn.execute(text(sql_ejecucion_c_dip_aper))
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_c_dip_aper (10201)", res.rowcount)
ejecucion_c_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_c_dip_aper"), work_conn)
_log("ejecucion_c_dip_aper", ejecucion_c_dip_aper)


In [ ]:
# /* Transacciones diferenciadas para coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip_aper"))
sql_ejecucion_d_dip_aper = f"""
SELECT T1.SUBSECTOR, T1.MONEDA, T1.AÑO,
    (CASE WHEN T1.MES IN (1,2,3) THEN 1
          WHEN T1.MES IN (4,5,6) THEN 2
          WHEN T1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    T1.MES, T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA,
    T1.C_TIPO_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION,
    T1.C_ANALITICO, T1.N_ANALITICO,
    T2.C_ACTIVO AS C_SCN, T2.C_CAGENTE, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA,
    T2.C_ACTIVO, T2.C_CUP,
    {caso_c_act} AS C_ACT,
    T1.NIVEL,
    CAST((T1.DEVENG * T3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_d_dip_aper
FROM #ejec_dip T1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 T2
  ON T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
 AND T1.C_TIPO_CUENTA = T2.T_CUENTA AND T1.C_CUENTA = T2.C_CUENTA AND T1.C_ITEM = T2.C_ITEM
 AND T1.C_ASIGNACION = T2.C_ASIGNACION AND T1.C_ANALITICO = T2.C_ANALITICO AND T1.C_ENTIDAD = T2.ENTIDAD
LEFT JOIN #prom_mes T3
  ON T1.MONEDA = T3.MONEDA AND T1.MES = T3.MES AND T1.AÑO = T3.AÑO
WHERE T2.OBS IN ('CR18_difcoy') AND T1.C_ENTIDAD NOT IN (5601,10201) AND T2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND T2.C_SCN NOT IN ('TC')
"""
work_conn.execute(text(sql_ejecucion_d_dip_aper))
ejecucion_d_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_d_dip_aper"), work_conn)
_log("ejecucion_d_dip_aper", ejecucion_d_dip_aper)


In [ ]:
# /* Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip_aper"))
sql_ejecucion_f_dip_aper = f"""
SELECT T1.SUBSECTOR, T1.MONEDA, T1.AÑO,
    (CASE WHEN T1.MES IN (1,2,3) THEN 1
          WHEN T1.MES IN (4,5,6) THEN 2
          WHEN T1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    T1.MES, T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA,
    T1.C_TIPO_CUENTA, T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION,
    T1.C_ANALITICO, T1.N_ANALITICO,
    T2.C_SCN, T2.C_CAGENTE, T2.C_TRANSACCION, T2.T_TRANSACCION, T2.N_CUENTA,
    T2.C_ACTIVO, T2.C_CUP,
    T2.C_CUP AS C_ACT,
    T1.NIVEL,
    CAST((T1.DEVENG * T3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_f_dip_aper
FROM #ejec_dip T1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 T2
  ON T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
 AND T1.C_TIPO_CUENTA = T2.T_CUENTA AND T1.C_CUENTA = T2.C_CUENTA AND T1.C_ITEM = T2.C_ITEM
 AND T1.C_ASIGNACION = T2.C_ASIGNACION AND T1.C_ANALITICO = T2.C_ANALITICO AND T1.C_ENTIDAD = T2.ENTIDAD
LEFT JOIN #prom_mes T3
  ON T1.MONEDA = T3.MONEDA AND T1.MES = T3.MES AND T1.AÑO = T3.AÑO
WHERE T2.OBS IN ('CR18_difact') AND T1.C_ENTIDAD NOT IN (5601,10201) AND T2.C_SCN NOT IN ('TC') AND T2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_ejecucion_f_dip_aper))
ejecucion_f_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_f_dip_aper"), work_conn)
_log("ejecucion_f_dip_aper", ejecucion_f_dip_aper)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip_aper"))
# DROP=ACTIVIDAD: ACTIVIDAD no se incluye en el SELECT final
sql_ejecucion_g_dip_aper = """
SELECT T1.SUBSECTOR, T1.MONEDA, T1.AÑO, T1.TRIM, T1.MES, T1.C_ENTIDAD,
    T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA, T1.C_CUENTA,
    T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO, T1.C_SCN,
    T1.C_CAGENTE, T1.C_TRANSACCION, T1.T_TRANSACCION, T1.N_CUENTA, T1.C_ACTIVO,
    T1.C_CUP,
    (CASE WHEN T1.ACTIVIDAD='FFAA' THEN 'AAPP'
          WHEN T1.ACTIVIDAD='FFOO' THEN 'AAPP'
          WHEN T1.ACTIVIDAD='PREVISION' THEN 'AAPP'
          WHEN T1.ACTIVIDAD='FONASA' THEN 'AAPP'
          WHEN T1.ACTIVIDAD='CORFO' THEN 'AAPP'
          WHEN T1.ACTIVIDAD='CENABAST' THEN 'AAPP'
          WHEN T1.ACTIVIDAD='AAPP' THEN 'AAPP'
          WHEN T1.ACTIVIDAD='SSPP' THEN 'SSPP'
          WHEN T1.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
    T1.NIVEL,
    T1.dato
INTO #ejecucion_g_dip_aper
FROM #ejec_g_mixt_aper_dip T1
WHERE T1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY T1.SUBSECTOR, T1.MONEDA, T1.AÑO, T1.TRIM, T1.MES, T1.C_ENTIDAD,
    T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA, T1.C_CUENTA,
    T1.C_ITEM, T1.C_SCN, T1.C_CAGENTE, T1.C_TRANSACCION, T1.T_TRANSACCION,
    T1.N_CUENTA, T1.C_ACTIVO, T1.C_CUP, T1.NIVEL, T1.dato,
    T1.ACTIVIDAD, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO
"""
# el GROUP BY original de SAS no incluye ACTIVIDAD (usada solo en el CASE) ni
# C_ASIGNACION/C_ANALITICO/N_ANALITICO (PROC SQL toma el primer registro del
# grupo); se agregan al GROUP BY porque T-SQL exige toda columna no agregada
work_conn.execute(text(sql_ejecucion_g_dip_aper))
ejecucion_g_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_g_dip_aper"), work_conn)
_log("ejecucion_g_dip_aper", ejecucion_g_dip_aper)


In [ ]:
# /*UNE TABLAS de Salida total de ejecución CG coy de data DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper"))
sql_ejecucion_cg_dip_aper = """
SELECT * INTO #ejecucion_cg_dip_aper FROM #ejecucion_a_dip_aper
UNION ALL
SELECT * FROM #ejecucion_b_dip_aper
UNION ALL
SELECT * FROM #ejecucion_c_dip_aper
UNION ALL
SELECT * FROM #ejecucion_d_dip_aper
UNION ALL
SELECT * FROM #ejecucion_g_dip_aper
UNION ALL
SELECT * FROM #ejecucion_f_dip_aper
"""
work_conn.execute(text(sql_ejecucion_cg_dip_aper))


In [ ]:
# DROP TABLE EJECUCION_A_PRELIM_DIP_APER, EJECUCION_A_DIP_aper, EJECUCION_B_DIP_aper,
# EJECUCION_C_DIP_aper, EJECUCION_D_DIP_aper, EJECUCION_G_DIP_aper, EJECUCION_F_DIP_aper
for _t in ["#ejecucion_a_prelim_dip_aper", "#ejecucion_a_dip_aper", "#ejecucion_b_dip_aper",
           "#ejecucion_c_dip_aper", "#ejecucion_d_dip_aper", "#ejecucion_g_dip_aper", "#ejecucion_f_dip_aper"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_aper SET N_CUENTA='DIPRES'"))
_log("UPDATE #ejecucion_cg_dip_aper (N_CUENTA)", res.rowcount)
ejecucion_cg_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_aper"), work_conn)
_log("ejecucion_cg_dip_aper", ejecucion_cg_dip_aper)


In [ ]:
# /*UNE TABLAS de Salida total de ejecución CG coy con info completa, CGR + DIPRES (mes t)*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_aper"))
sql_ejecucion_cg_aper = """
SELECT * INTO #ejecucion_cg_aper FROM #ejecucion_cg_cgr_aper
UNION ALL
SELECT * FROM #ejecucion_cg_dip_aper
"""
work_conn.execute(text(sql_ejecucion_cg_aper))
ejecucion_cg_aper = pd.read_sql(text("SELECT * FROM #ejecucion_cg_aper"), work_conn)
_log("ejecucion_cg_aper", ejecucion_cg_aper)


In [ ]:
# DROP TABLE EJECUCION_DIP_aper, CR18_APERTURA_PACAPR (EJECUCION_CG_CGR_aper queda comentado en el SAS: se conserva)
for _t in ["#ejecucion_dip_aper", "#cr18_apertura_pacapr"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 2- IMPUT TC Mixtas_M_DIP

Imputa las transacciones de tipo Mixtas (M) de la Ejecución DIPRES faltantes en la muestra, redistribuyendo el gasto devengado según la estructura de ponderación de muestra y expandiéndolo al resto de analíticos fuera de muestra, para entregar la ejecución total (muestra + resto) desagregada y consolidada por transacción SCN

*confianza: medium · verificador: approve · SAS: PROC SQL: CREATE TABLE con LEFT JOIN, CASE dentro de SUM, GROUP BY, UNION ALL y DROP TABLE de limpieza*

In [ ]:
# ========= 2- IMPUT TC Mixtas_M_DIP =========
# /*Ejecucion dipres del periodo, con entidad asignada*/
# /*MODIFICADO JUNIO 2025*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip_y"))
sql_ejec_dip_y = f"""
SELECT T1.*, T2.C_ENTIDAD, T2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip_y
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
    ON T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA
   AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejec_dip_y))
ejec_dip_y = pd.read_sql(text("SELECT * FROM #ejec_dip_y"), work_conn)
_log("ejec_dip_y", ejec_dip_y)


In [ ]:
# /* IMPUTACIONES: CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MIXTAS (M) SEGUN ESTRUCTURA, PARA EJECUCION AÑO, MES y PACAPR faltante */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muest_mixtas_m_prev_dip_imp"))
sql_ejec_muest_prev = f"""
SELECT t2.*,
       t1.C_SCN,
       t1.C_CAGENTE,
       CAST(t1.C_CUP AS char(4)) AS C_CUP,
       (CASE WHEN t1.C_SCN = 'D11' THEN SUM(t2.DEVENG/1000000.0*t1.POND)
             WHEN t1.C_SCN = 'P2'  THEN SUM(t2.DEVENG/1000000.0*t1.POND)
             WHEN t1.C_SCN = 'P51' THEN SUM(t2.DEVENG/1000000.0*t1.POND)
             WHEN t1.C_SCN = 'D75' THEN SUM(t2.DEVENG/1000000.0*t1.POND) END) AS DEVENGO
INTO #ejec_muest_mixtas_m_prev_dip_imp
FROM GOBGENER.dbo.EST_MIXTAS_M t1
LEFT JOIN #ejec_dip_y t2
    ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM AND t2.MES = t1.MES
WHERE t2.C_CUENTA IN ('24','33') AND t2.C_ITEM IN ('03','01','06','09','08')
  AND t2.MES IN ({Mes_imp1},{Mes_imp2},{Mes_imp3},{Mes_imp4},{Mes_imp5},{Mes_imp6},{Mes_imp7},{Mes_imp8},{Mes_imp9},{Mes_imp10},{Mes_imp11},{Mes_imp12})
  AND t1.ENTIDAD IN ({PACAPR1},{PACAPR2},{PACAPR3},{PACAPR4},{PACAPR5},{PACAPR6},{PACAPR7})
  AND t2.MONEDA IN ('{Moneda}')
GROUP BY t2.MES, t2.AÑO, T2.C_ENTIDAD, t1.C_SCN, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
         t1.C_ANALITICO, t1.C_CAGENTE, CAST(t1.C_CUP AS char(4)), T2.ACTIVIDAD, T2.NIVEL
"""
work_conn.execute(text(sql_ejec_muest_prev))


In [ ]:
# DATA step: DROP de columnas de apoyo, mismo dataset renombrado (ejecutable) DROP TABLE EJEC_MUEST_MIXTAS_M_prev_DIP_imp
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muest_mixtas_m_dip_imp"))
sql_ejec_muest_dip_imp = """
SELECT * INTO #ejec_muest_mixtas_m_dip_imp
FROM #ejec_muest_mixtas_m_prev_dip_imp
"""
work_conn.execute(text(sql_ejec_muest_dip_imp))
work_conn.execute(text("ALTER TABLE #ejec_muest_mixtas_m_dip_imp DROP COLUMN PPTO_INICIAL, PPTO_ACTUAL, DEVENG, EJEC_ACUM"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muest_mixtas_m_prev_dip_imp"))
ejec_muest_mixtas_m_dip_imp = pd.read_sql(text("SELECT * FROM #ejec_muest_mixtas_m_dip_imp"), work_conn)
_log("ejec_muest_mixtas_m_dip_imp", ejec_muest_mixtas_m_dip_imp)


In [ ]:
# /* IMPUTACIONES: CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m_dip_imp"))
sql_cruce_ejec_exp = f"""
SELECT t2.*,
       t1.C_CUENTA,
       t1.C_ITEM,
       T1.ENTIDAD,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m_dip_imp
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_dip_y t2
    ON t2.C_ENTIDAD = T1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.C_CUENTA = t1.C_CUENTA
   AND t2.C_ITEM = t1.C_ITEM
WHERE t2.C_CUENTA IN ('24','33') AND t2.C_ITEM IN ('03','01','06','09','08')
  AND t2.MES IN ({Mes_imp1},{Mes_imp2},{Mes_imp3},{Mes_imp4},{Mes_imp5},{Mes_imp6},{Mes_imp7},{Mes_imp8},{Mes_imp9},{Mes_imp10},{Mes_imp11},{Mes_imp12})
  AND t2.MONEDA IN ('{Moneda}')
  -- AND T1.C_ENTIDAD IN (&PACAPR) -- comentado en el SAS original, se mantiene anulado
  AND t1.M_MUESTRA = 0 AND t1.Clasif_Tcons = 'M'
GROUP BY t2.AÑO, T2.MES, t1.C_CUENTA, t1.C_ITEM, T1.ENTIDAD, t1.C_ASIGNACION, t1.C_ANALITICO, t1.Clasif_Tcons
"""
work_conn.execute(text(sql_cruce_ejec_exp))
cruce_ejec_exp_mixtas_m_dip_imp = pd.read_sql(text("SELECT * FROM #cruce_ejec_exp_mixtas_m_dip_imp"), work_conn)
_log("cruce_ejec_exp_mixtas_m_dip_imp", cruce_ejec_exp_mixtas_m_dip_imp)


In [ ]:
# /* IMPUTACIONES: CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M), PARA EJECUCION AÑO, MES y PACAPR faltante */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim_dip_imp"))
sql_ejec_exp_prelim = f"""
SELECT t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD, t2.C_PARTIDA, t2.C_CAPITULO,
       t2.C_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
       t2.N_ANALITICO, t2.PPTO_INICIAL, t2.PPTO_ACTUAL, t2.DEVENG, t2.Clasif_Tcons,
       t1.C_SCN, t1.C_CAGENTE, t1.C_CUP,
       SUM(t2.DEVENG) AS DATO_OR,
       (CASE WHEN t1.C_SCN = 'D11' THEN SUM(t2.DEVENG/1000000.0*t1.POND*t3.DATO)
             WHEN t1.C_SCN = 'P2'  THEN SUM(t2.DEVENG/1000000.0*t1.POND*t3.DATO)
             WHEN t1.C_SCN = 'P51' THEN SUM(t2.DEVENG/1000000.0*t1.POND*t3.DATO)
             WHEN t1.C_SCN = 'D75' THEN SUM(t2.DEVENG/1000000.0*t1.POND*t3.DATO)
             WHEN t1.C_SCN = 'D73' THEN SUM(t2.DEVENG/1000000.0*t1.POND*t3.DATO)
             WHEN t1.C_SCN = 'D63' THEN SUM(t2.DEVENG/1000000.0*t1.POND*t3.DATO) END) AS DEVENGO
INTO #ejec_exp_mixtas_m_prelim_dip_imp
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M t1
LEFT JOIN #cruce_ejec_exp_mixtas_m_dip_imp t2
    ON t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM AND t2.C_ASIGNACION = t1.C_ASIGNACION
   AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.MES = t1.MES AND t2.C_ENTIDAD = t1.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t2.MONEDA = t3.MONEDA AND t2.MES = t3.MES AND t2.AÑO = t3.AÑO
   AND t2.MES IN ({Mes_imp1},{Mes_imp2},{Mes_imp3},{Mes_imp4},{Mes_imp5},{Mes_imp6},{Mes_imp7},{Mes_imp8},{Mes_imp9},{Mes_imp10},{Mes_imp11},{Mes_imp12})
WHERE t2.C_CUENTA IN ('24','33') AND t2.C_ITEM IN ('03','01','06','09','08')
  AND t2.C_ENTIDAD IN ({PACAPR1},{PACAPR2},{PACAPR3},{PACAPR4},{PACAPR5},{PACAPR6},{PACAPR7})
GROUP BY t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_ENTIDAD, t2.C_PARTIDA, t2.C_CAPITULO,
         t2.C_PROGRAMA, t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO,
         t2.N_ANALITICO, t2.PPTO_INICIAL, t2.PPTO_ACTUAL, t2.DEVENG, t2.Clasif_Tcons,
         t1.C_SCN, t1.C_CAGENTE, t1.C_CUP
"""
work_conn.execute(text(sql_ejec_exp_prelim))


In [ ]:
# /* HOMOLOGACION TABLA DE EJECUCION DE RESTO DE TRANSACCIONES MIXTAS (M) EXPANDIDA A MUESTRA, ACORDE A TABLA EJECUCION MUESTRA */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_dip_imp"))
sql_ejec_exp_dip_imp = """
SELECT * INTO #ejec_exp_mixtas_m_dip_imp
FROM #ejec_exp_mixtas_m_prelim_dip_imp
"""
work_conn.execute(text(sql_ejec_exp_dip_imp))
work_conn.execute(text("ALTER TABLE #ejec_exp_mixtas_m_dip_imp DROP COLUMN PPTO_INICIAL, PPTO_ACTUAL, DEVENG, EJEC_ACUM, DATO_OR, Clasif_Tcons"))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim_dip_imp"))
ejec_exp_mixtas_m_dip_imp = pd.read_sql(text("SELECT * FROM #ejec_exp_mixtas_m_dip_imp"), work_conn)
_log("ejec_exp_mixtas_m_dip_imp", ejec_exp_mixtas_m_dip_imp)


In [ ]:
# /* Crea tabla con la ejecución total de transacciones Mixtas (M), Muestra + Resto */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total_dip_imp"))
sql_ejec_mixtas_m_total = """
SELECT * INTO #ejec_mixtas_m_total_dip_imp FROM #ejec_muest_mixtas_m_dip_imp
UNION ALL
SELECT * FROM #ejec_exp_mixtas_m_dip_imp
"""
work_conn.execute(text(sql_ejec_mixtas_m_total))
ejec_mixtas_m_total_dip_imp = pd.read_sql(text("SELECT * FROM #ejec_mixtas_m_total_dip_imp"), work_conn)
_log("ejec_mixtas_m_total_dip_imp", ejec_mixtas_m_total_dip_imp)


In [ ]:
# /* IMPUTACIONES: SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CG coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_z_mixt_dip_imp"))
sql_ejec_z_mixt = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE,
       t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t1.C_CUP,
       SUM(t1.DEVENGO) AS Dato, t1.ACTIVIDAD, t1.NIVEL, t1.C_ENTIDAD AS [KEY]
INTO #ejec_z_mixt_dip_imp
FROM #ejec_mixtas_m_total_dip_imp t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_SCN IN ('D11','P2','D63')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL,
         t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t1.C_CUP
"""
work_conn.execute(text(sql_ejec_z_mixt))
ejec_z_mixt_dip_imp = pd.read_sql(text("SELECT * FROM #ejec_z_mixt_dip_imp"), work_conn)
_log("ejec_z_mixt_dip_imp", ejec_z_mixt_dip_imp)


In [ ]:
# /* IMPUTACIONES: SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para ANALISIS DESAGREGADO CG coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_z_mixt_aper_dip_imp"))
sql_ejec_z_mixt_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
       t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN,
       t1.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       SUM(t1.DEVENGO) AS Dato, t1.ACTIVIDAD, t1.NIVEL
INTO #ejec_z_mixt_aper_dip_imp
FROM #ejec_mixtas_m_total_dip_imp t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_SCN IN ('D11','P2','D63')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
         (CASE WHEN t1.MES IN (1,2,3) THEN 1
               WHEN t1.MES IN (4,5,6) THEN 2
               WHEN t1.MES IN (7,8,9) THEN 3
               ELSE 4 END),
         t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
         t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN,
         t1.C_CAGENTE, t1.ACTIVIDAD, t1.NIVEL,
         t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP
"""
work_conn.execute(text(sql_ejec_z_mixt_aper))
ejec_z_mixt_aper_dip_imp = pd.read_sql(text("SELECT * FROM #ejec_z_mixt_aper_dip_imp"), work_conn)
_log("ejec_z_mixt_aper_dip_imp", ejec_z_mixt_aper_dip_imp)


In [ ]:
# PROC SQL DROP TABLE final: limpia temporales intermedias de la sesión (EST_MIXTAS_M_prev_DIP_imp y ANALIT_TCONS_prev_DIP_imp no se llegaron a crear en este nodo; se ignoran)
for _t in ["#ejec_muest_mixtas_m_prev_dip_imp", "#ejec_muest_mixtas_m_dip_imp",
           "#cruce_ejec_exp_mixtas_m_dip_imp", "#ejec_exp_mixtas_m_prelim_dip_imp",
           "#ejec_exp_mixtas_m_dip_imp", "#ejec_mixtas_m_total_dip_imp", "#ejec_dip_y"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 3- IMPUT Regulares DIP

Imputaciones regulares de gasto de gobierno central desde datos DIPRES del mes, clasificadas por analítico predefinido/variable/diferenciado y consolidadas en una sola tabla de imputaciones

*confianza: medium · verificador: approve · SAS: PROC SQL: creación de #tmp encadenadas con LEFT JOIN, CASE WHEN, cálculo de KEY numérico vía compress/input/CATX, agregación GROUP BY, UNION ALL y UPDATE final*

In [ ]:
# ========= 3- IMPUT Regulares DIP =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip_y"))
# /*Ejecucion dipres del periodo, con entidad asignada*/
sql_ejec_dip_y = f"""
SELECT T1.*, T2.C_ENTIDAD, T2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip_y
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
    ON T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA
    AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejec_dip_y))
ejec_dip_y = pd.read_sql(text("SELECT * FROM #ejec_dip_y"), work_conn)
_log("ejec_dip_y", ejec_dip_y)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_imp"))
# /*IMPUTACIONES:Analiticos predefinidos*/
# lista de meses de imputación como literal controlado (macro params &Mes_impN)
meses_imp = f"{Mes_imp1},{Mes_imp2},{Mes_imp3},{Mes_imp4},{Mes_imp5},{Mes_imp6},{Mes_imp7},{Mes_imp8},{Mes_imp9},{Mes_imp10},{Mes_imp11},{Mes_imp12}"
sql_ejecucion_a_prelim_dip_imp = f"""
SELECT t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
    t2.C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t2.C_ACTIVO,
    t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS DEVENGO,
    TRY_CAST(REPLACE(CONCAT('00', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', '') AS float) AS [KEY]
INTO #ejecucion_a_prelim_dip_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t1.MES IN ({meses_imp}) AND t1.MONEDA = '{Moneda}'
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip_imp))


In [ ]:
# /*IMPUTACIONES:Analiticos predefinidos*/ -- filtros de exclusión puntual
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_imp WHERE [KEY] = 10201"))
_log("DELETE #ejecucion_a_prelim_dip_imp (KEY=10201)", res.rowcount)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_imp WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_dip_imp (C_CUENTA=32)", res.rowcount)
# /*AJUSTE COYUNTURA - se elimina de indicador transacción CORFO que se desagrega anualmente entre ventas y D45*/
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_imp WHERE [KEY] = 3801 AND C_CUENTA = '06' AND C_ITEM = '01' AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')"))
_log("DELETE #ejecucion_a_prelim_dip_imp (KEY=3801 CORFO)", res.rowcount)
ejecucion_a_prelim_dip_imp = pd.read_sql(text("SELECT * FROM #ejecucion_a_prelim_dip_imp"), work_conn)
_log("ejecucion_a_prelim_dip_imp", ejecucion_a_prelim_dip_imp)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_imp"))
# /*IMPUTACIONES:Analiticos predefinidos (sigue)*/
pacapr = f"{PACAPR1},{PACAPR2},{PACAPR3},{PACAPR4},{PACAPR5},{PACAPR6},{PACAPR7}"
sql_ejecucion_a_dip_imp = f"""
SELECT t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    t1.TRIM,
    t1.MES,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_SCN,
    t1.C_CAGENTE,
    t1.C_TRANSACCION,
    t1.T_TRANSACCION,
    t1.N_CUENTA,
    t1.C_ACTIVO,
    t1.C_CUP,
    t1.C_ACT,
    t1.NIVEL,
    SUM(t1.DEVENGO) AS Dato,
    t1.[KEY]
INTO #ejecucion_a_dip_imp
FROM #ejecucion_a_prelim_dip_imp t1
WHERE t1.[KEY] IN ({pacapr}) AND t1.MONEDA = '{Moneda}'
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
    t1.C_ACT, t1.NIVEL, t1.[KEY]
"""
work_conn.execute(text(sql_ejecucion_a_dip_imp))
ejecucion_a_dip_imp = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip_imp"), work_conn)
_log("ejecucion_a_dip_imp", ejecucion_a_dip_imp)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip_imp"))
# /*IMPUTACIONES:Analiticos variables*/
sql_ejecucion_b_dip_imp = f"""
SELECT t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t2.C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t2.C_ACTIVO,
    t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato,
    TRY_CAST(REPLACE(CONCAT('00', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', '') AS float) AS [KEY]
INTO #ejecucion_b_dip_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t1.C_ANALITICO IN ('04001000','04002000')
    AND t1.MES IN ({meses_imp}) AND t1.MONEDA = '{Moneda}'
"""
work_conn.execute(text(sql_ejecucion_b_dip_imp))
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip_imp WHERE [KEY] = 1500040001"))
_log("DELETE #ejecucion_b_dip_imp (KEY=1500040001)", res.rowcount)
res = work_conn.execute(text(f"DELETE FROM #ejecucion_b_dip_imp WHERE [KEY] NOT IN ({pacapr}) AND MONEDA = '{Moneda}'"))
_log("DELETE #ejecucion_b_dip_imp (KEY NOT IN PACAPR)", res.rowcount)
ejecucion_b_dip_imp = pd.read_sql(text("SELECT * FROM #ejecucion_b_dip_imp"), work_conn)
_log("ejecucion_b_dip_imp", ejecucion_b_dip_imp)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_imp"))
# /*IMPUTACIONES: analiticos predefinidos que se filtran por CGR/actividad*/
sql_ejecucion_c_dip_imp = f"""
SELECT t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t2.C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t2.C_ACTIVO,
    t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato,
    TRY_CAST(REPLACE(CONCAT('00', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', '') AS float) AS [KEY]
INTO #ejecucion_c_dip_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t2.C_SCN <> 'TC'
    AND t1.MES IN ({meses_imp}) AND t1.MONEDA = '{Moneda}'
"""
work_conn.execute(text(sql_ejecucion_c_dip_imp))
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip_imp WHERE [KEY] = 1500040001"))
_log("DELETE #ejecucion_c_dip_imp (KEY=1500040001)", res.rowcount)
res = work_conn.execute(text(f"DELETE FROM #ejecucion_c_dip_imp WHERE [KEY] NOT IN ({pacapr}) AND MONEDA = '{Moneda}'"))
_log("DELETE #ejecucion_c_dip_imp (KEY NOT IN PACAPR)", res.rowcount)
ejecucion_c_dip_imp = pd.read_sql(text("SELECT * FROM #ejecucion_c_dip_imp"), work_conn)
_log("ejecucion_c_dip_imp", ejecucion_c_dip_imp)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip_imp"))
# /*IMPUTACIONES:Transacciones diferenciadas para coyuntura*/
sql_ejecucion_d_dip_imp = f"""
SELECT t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t2.C_ACTIVO AS C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t2.C_ACTIVO,
    t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato,
    TRY_CAST(REPLACE(CONCAT('00', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', '') AS float) AS [KEY]
INTO #ejecucion_d_dip_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difcoy') AND t1.C_ENTIDAD NOT IN (5601, 10201) AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t2.C_SCN <> 'TC' AND t1.MES IN ({meses_imp}) AND t1.MONEDA = '{Moneda}'
"""
work_conn.execute(text(sql_ejecucion_d_dip_imp))
res = work_conn.execute(text(f"DELETE FROM #ejecucion_d_dip_imp WHERE [KEY] NOT IN ({pacapr}) AND MONEDA = '{Moneda}'"))
_log("DELETE #ejecucion_d_dip_imp (KEY NOT IN PACAPR)", res.rowcount)
ejecucion_d_dip_imp = pd.read_sql(text("SELECT * FROM #ejecucion_d_dip_imp"), work_conn)
_log("ejecucion_d_dip_imp", ejecucion_d_dip_imp)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip_imp"))
# /*IMPUTACIONES:Transacciones diferenciadas por actividad*/
sql_ejecucion_f_dip_imp = f"""
SELECT t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t2.C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t2.C_ACTIVO,
    t2.C_CUP,
    t2.C_CUP AS C_ACT,
    t1.NIVEL,
    CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato,
    TRY_CAST(REPLACE(CONCAT('00', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', '') AS float) AS [KEY]
INTO #ejecucion_f_dip_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difact') AND t2.C_SCN <> 'TC' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t1.MES IN ({meses_imp}) AND t1.MONEDA = '{Moneda}'
"""
work_conn.execute(text(sql_ejecucion_f_dip_imp))
res = work_conn.execute(text(f"DELETE FROM #ejecucion_f_dip_imp WHERE [KEY] NOT IN ({pacapr}) AND MONEDA = '{Moneda}'"))
_log("DELETE #ejecucion_f_dip_imp (KEY NOT IN PACAPR)", res.rowcount)
ejecucion_f_dip_imp = pd.read_sql(text("SELECT * FROM #ejecucion_f_dip_imp"), work_conn)
_log("ejecucion_f_dip_imp", ejecucion_f_dip_imp)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip_imp"))
# EJEC_Z_MIXT_DIP_imp lo dejó CodeTask-QMEyeoD7jJUN9nQu como WORK.EJEC_Z_MIXT_DIP_IMP (ya está en la sesión como #ejec_z_mixt_dip_imp si sigue la misma estrategia)
sql_ejecucion_g_dip_imp = """
SELECT t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    t1.TRIM,
    t1.MES,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_SCN,
    t1.C_CAGENTE,
    t1.C_TRANSACCION,
    t1.T_TRANSACCION,
    t1.N_CUENTA,
    t1.C_ACTIVO,
    t1.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    t1.DATO,
    TRY_CAST(REPLACE(CONCAT('00', t1.MONEDA), ' ', '') AS float) AS [KEY]
INTO #ejecucion_g_dip_imp
FROM #ejec_z_mixt_dip_imp t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
    t1.ACTIVIDAD, t1.NIVEL, t1.DATO
"""
work_conn.execute(text(sql_ejecucion_g_dip_imp))
ejecucion_g_dip_imp = pd.read_sql(text("SELECT * FROM #ejecucion_g_dip_imp"), work_conn)
_log("ejecucion_g_dip_imp", ejecucion_g_dip_imp)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imputaciones"))
# /*IMPUTACIONES:UNE TABLAS de Salida total de ejecucion CG coy de data DIPRES*/
sql_ejecucion_cg_dip_imputaciones = """
SELECT * INTO #ejecucion_cg_dip_imputaciones FROM #ejecucion_a_dip_imp
UNION ALL
SELECT * FROM #ejecucion_b_dip_imp
UNION ALL
SELECT * FROM #ejecucion_c_dip_imp
UNION ALL
SELECT * FROM #ejecucion_d_dip_imp
UNION ALL
SELECT * FROM #ejecucion_g_dip_imp
UNION ALL
SELECT * FROM #ejecucion_f_dip_imp
"""
work_conn.execute(text(sql_ejecucion_cg_dip_imputaciones))
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_imputaciones SET N_CUENTA = 'Imputacion'"))
_log("UPDATE #ejecucion_cg_dip_imputaciones (N_CUENTA)", res.rowcount)
ejecucion_cg_dip_imputaciones = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_imputaciones"), work_conn)
_log("ejecucion_cg_dip_imputaciones", ejecucion_cg_dip_imputaciones)


In [ ]:
# El SAS dropeaba las tablas intermedias al final; con la sesión el DROP TABLE IF EXISTS
# al inicio de cada bloque cumple ese rol y hace las celdas re-ejecutables.
# EJECUCION_A_PRELIM_DIP_imp y EJEC_DIP_Y quedan vivas: son output_datasets del nodo.
for _t in ["#ejecucion_a_dip_imp", "#ejecucion_b_dip_imp", "#ejecucion_c_dip_imp", "#ejecucion_d_dip_imp", "#ejecucion_g_dip_imp", "#ejecucion_f_dip_imp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## OPC Agregar IMPUT a permanentes*

Actualiza las imputaciones permanentes de gasto (agregado y desagregado por partida) reemplazando las combinaciones ya existentes con las recién calculadas del periodo

*confianza: medium · verificador: approve · SAS: PROC SQL DELETE EXISTS correlacionado + PROC DATASETS APPEND FORCE + DROP TABLE*

In [ ]:
# ========= OPC Agregar IMPUT a permanentes* =========
# /*ACTUALIZACION DE LAS IMPUTACIONES PERMANENTES (Sólo si corresponden a permanentes!) */
# /* para salida de indicadores*/
# EJECUCION_CG_DIP_imputaciones (WORK) queda materializada como #tmp por el nodo anterior (session_sql)
sql_delete_imput_gc_dip = """
DELETE FROM GOBGENER.dbo.IMPUT_GC_DIP
WHERE EXISTS (
    SELECT 1 FROM #ejecucion_cg_dip_imputaciones e
    WHERE e.SUBSECTOR = IMPUT_GC_DIP.SUBSECTOR AND e.MONEDA = IMPUT_GC_DIP.MONEDA
    AND e.AÑO = IMPUT_GC_DIP.AÑO AND e.TRIM = IMPUT_GC_DIP.TRIM
    AND e.MES = IMPUT_GC_DIP.MES AND e.C_TIPO_CUENTA = IMPUT_GC_DIP.C_TIPO_CUENTA
    AND e.C_CUENTA = IMPUT_GC_DIP.C_CUENTA AND e.C_ITEM = IMPUT_GC_DIP.C_ITEM
    AND e.C_SCN = IMPUT_GC_DIP.C_SCN AND e.C_CAGENTE = IMPUT_GC_DIP.C_CAGENTE
    AND e.C_TRANSACCION = IMPUT_GC_DIP.C_TRANSACCION
    AND e.T_TRANSACCION = IMPUT_GC_DIP.T_TRANSACCION AND
    e.C_ACTIVO = IMPUT_GC_DIP.C_ACTIVO AND e.C_CUP = IMPUT_GC_DIP.C_CUP
    AND e.C_ACT = IMPUT_GC_DIP.C_ACT AND e.[KEY] = IMPUT_GC_DIP.[KEY]
    AND e.NIVEL = IMPUT_GC_DIP.NIVEL
)
"""
res = work_conn.execute(text(sql_delete_imput_gc_dip))
_log("DELETE GOBGENER.dbo.IMPUT_GC_DIP", res.rowcount)


In [ ]:
# PROC DATASETS APPEND FORCE: inserta TODAS las columnas del dataset origen
# (FORCE tolera diferencias de estructura), no solo las de la clave del EXISTS.
cols_ejecucion_cg_dip_imputaciones = pd.read_sql(
    text("SELECT TOP 0 * FROM #ejecucion_cg_dip_imputaciones"), work_conn
).columns.tolist()
cols_sql_imput_gc_dip = ", ".join(f"[{c}]" for c in cols_ejecucion_cg_dip_imputaciones)
sql_append_imput_gc_dip = f"""
INSERT INTO GOBGENER.dbo.IMPUT_GC_DIP ({cols_sql_imput_gc_dip})
SELECT {cols_sql_imput_gc_dip}
FROM #ejecucion_cg_dip_imputaciones
"""
res = work_conn.execute(text(sql_append_imput_gc_dip))
_log("APPEND GOBGENER.dbo.IMPUT_GC_DIP", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imputaciones"))


In [ ]:
# /* para análisis desagregado*/
# EJECUCION_CG_DIP_APER_IMP (WORK) queda materializada como #tmp por el nodo anterior (session_sql)
sql_delete_imput_gc_dip_aper = """
DELETE FROM GOBGENER.dbo.IMPUT_GC_DIP_APER
WHERE EXISTS (
    SELECT 1 FROM #ejecucion_cg_dip_aper_imp e
    WHERE e.SUBSECTOR = IMPUT_GC_DIP_APER.SUBSECTOR AND e.MONEDA = IMPUT_GC_DIP_APER.MONEDA
    AND e.AÑO = IMPUT_GC_DIP_APER.AÑO AND e.TRIM = IMPUT_GC_DIP_APER.TRIM
    AND e.MES = IMPUT_GC_DIP_APER.MES AND e.C_ENTIDAD = IMPUT_GC_DIP_APER.C_ENTIDAD
    AND e.C_PARTIDA = IMPUT_GC_DIP_APER.C_PARTIDA AND e.C_CAPITULO = IMPUT_GC_DIP_APER.C_CAPITULO
    AND e.C_PROGRAMA = IMPUT_GC_DIP_APER.C_PROGRAMA AND e.C_TIPO_CUENTA = IMPUT_GC_DIP_APER.C_TIPO_CUENTA
    AND e.C_CUENTA = IMPUT_GC_DIP_APER.C_CUENTA AND e.C_ITEM = IMPUT_GC_DIP_APER.C_ITEM AND
    e.C_ASIGNACION = IMPUT_GC_DIP_APER.C_ASIGNACION AND e.C_ANALITICO = IMPUT_GC_DIP_APER.C_ANALITICO
    AND e.N_ANALITICO = IMPUT_GC_DIP_APER.N_ANALITICO AND e.C_SCN = IMPUT_GC_DIP_APER.C_SCN
    AND e.C_CAGENTE = IMPUT_GC_DIP_APER.C_CAGENTE AND e.C_TRANSACCION = IMPUT_GC_DIP_APER.C_TRANSACCION
    AND e.T_TRANSACCION = IMPUT_GC_DIP_APER.T_TRANSACCION AND e.C_ACTIVO = IMPUT_GC_DIP_APER.C_ACTIVO
    AND e.C_CUP = IMPUT_GC_DIP_APER.C_CUP AND e.C_ACT = IMPUT_GC_DIP_APER.C_ACT AND e.NIVEL = IMPUT_GC_DIP_APER.NIVEL
)
"""
res = work_conn.execute(text(sql_delete_imput_gc_dip_aper))
_log("DELETE GOBGENER.dbo.IMPUT_GC_DIP_APER", res.rowcount)


In [ ]:
# PROC DATASETS APPEND FORCE: inserta TODAS las columnas del dataset origen
# (FORCE tolera diferencias de estructura), no solo las de la clave del EXISTS.
cols_ejecucion_cg_dip_aper_imp = pd.read_sql(
    text("SELECT TOP 0 * FROM #ejecucion_cg_dip_aper_imp"), work_conn
).columns.tolist()
cols_sql_imput_gc_dip_aper = ", ".join(f"[{c}]" for c in cols_ejecucion_cg_dip_aper_imp)
sql_append_imput_gc_dip_aper = f"""
INSERT INTO GOBGENER.dbo.IMPUT_GC_DIP_APER ({cols_sql_imput_gc_dip_aper})
SELECT {cols_sql_imput_gc_dip_aper}
FROM #ejecucion_cg_dip_aper_imp
"""
res = work_conn.execute(text(sql_append_imput_gc_dip_aper))
_log("APPEND GOBGENER.dbo.IMPUT_GC_DIP_APER", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_imp"))


## IMPUT DIP Aper_CG_coy

Ejecución DIPRES desagregada por entidad imputando analíticos predefinidos, variables y transacciones diferenciadas por coyuntura/actividad, consolidada en una única tabla marcada como Imputación

*confianza: medium · verificador: revise · SAS: PROC SQL: LEFT JOINs de imputación con GROUP BY parcial (CASE + agregación tipo 'primer registro'), múltiples DELETE, UNION ALL y UPDATE final*

In [ ]:
# ========= IMPUT DIP Aper_CG_coy =========
# /* IMPUTACIONES: Salida para ANALISIS DESAGREGADO de indicadores GC de consumo de gobierno (coyuntura) */
# /*MODIFICADO JUNIO 2025*/
# Ejecucion dipres del periodo, con entidad asignada
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip_y"))
sql_ejec_dip_y = f"""
SELECT T1.*, T2.C_ENTIDAD, t2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip_y
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
  ON (T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA)
WHERE t1.AÑO = {int(Year)}
"""
work_conn.execute(text(sql_ejec_dip_y))
ejec_dip_y = pd.read_sql(text("SELECT * FROM #ejec_dip_y"), work_conn)
_log("ejec_dip_y", ejec_dip_y)


In [ ]:
# /*IMPUTACIONES:Analiticos predefinidos */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper_imp"))
sql_ejecucion_a_prelim_dip_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       T1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON (t1.C_TIPO_CUENTA = t2.T_CUENTA) AND (t1.C_CUENTA = t2.C_CUENTA) AND (t1.C_ANALITICO = t2.C_ANALITICO)
LEFT JOIN #prom_mes t3
  ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)}
  AND t1.MES IN ({int(Mes_imp1)},{int(Mes_imp2)},{int(Mes_imp3)},{int(Mes_imp4)},{int(Mes_imp5)},{int(Mes_imp6)},{int(Mes_imp7)},{int(Mes_imp8)},{int(Mes_imp9)},{int(Mes_imp10)},{int(Mes_imp11)},{int(Mes_imp12)})
  AND t1.MONEDA IN ('{Moneda}')
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip_aper_imp))


In [ ]:
# DELETE FROM EJECUCION_A_prelim_DIP_aper_imp WHERE C_ENTIDAD IN (1500040001)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper_imp WHERE C_ENTIDAD IN (1500040001)"))
_log("DELETE #ejecucion_a_prelim_dip_aper_imp (entidad 1500040001)", res.rowcount)


In [ ]:
# DELETE FROM EJECUCION_A_prelim_DIP_aper_imp WHERE C_CUENTA='32' AND C_ANALITICO IN (...)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper_imp WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_dip_aper_imp (cuenta 32)", res.rowcount)


In [ ]:
# /*IMPUTACIONES:Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper_imp WHERE C_ENTIDAD IN (3801) AND C_CUENTA = '06' AND C_ITEM = '01' AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')"))
_log("DELETE #ejecucion_a_prelim_dip_aper_imp (ajuste CORFO)", res.rowcount)
ejecucion_a_prelim_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_a_prelim_dip_aper_imp"), work_conn)
_log("ejecucion_a_prelim_dip_aper_imp", ejecucion_a_prelim_dip_aper_imp)


In [ ]:
# /*IMPUTACIONES:Analiticos predefinidos (sigue)*/
# GROUP BY exactamente como en el SAS: no incluye C_PARTIDA, C_CAPITULO, C_PROGRAMA,
# C_ASIGNACION, N_ANALITICO ni DEVENGO aunque están en el SELECT -> PROC SQL toma
# el valor del primer registro del grupo para esas columnas (colapsa filas)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_aper_imp"))
sql_ejecucion_a_dip_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       t1.TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       MIN(t1.C_PARTIDA) AS C_PARTIDA,
       MIN(t1.C_CAPITULO) AS C_CAPITULO,
       MIN(t1.C_PROGRAMA) AS C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       MIN(t1.C_ASIGNACION) AS C_ASIGNACION,
       t1.C_ANALITICO,
       MIN(t1.N_ANALITICO) AS N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_TRANSACCION,
       t1.T_TRANSACCION,
       t1.N_CUENTA,
       t1.C_ACTIVO,
       t1.C_CUP,
       t1.C_ACT,
       t1.NIVEL,
       MIN(t1.DEVENGO) AS Dato
INTO #ejecucion_a_dip_aper_imp
FROM #ejecucion_a_prelim_dip_aper_imp t1
WHERE t1.C_ENTIDAD IN ({int(PACAPR1)},{int(PACAPR2)},{int(PACAPR3)},{int(PACAPR4)},{int(PACAPR5)},{int(PACAPR6)},{int(PACAPR7)}) AND t1.MONEDA IN ('{Moneda}')
GROUP BY
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    t1.TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_SCN,
    t1.C_CAGENTE,
    t1.C_TRANSACCION,
    t1.T_TRANSACCION,
    t1.N_CUENTA,
    t1.C_ACTIVO,
    t1.C_CUP,
    t1.C_ACT,
    t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a_dip_aper_imp))
ejecucion_a_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip_aper_imp"), work_conn)
_log("ejecucion_a_dip_aper_imp", ejecucion_a_dip_aper_imp)


In [ ]:
# /*IMPUTACIONES:Analiticos variables */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip_aper_imp"))
sql_ejecucion_b_dip_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       T1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_b_dip_aper_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND T1.C_ENTIDAD = T2.ENTIDAD)
LEFT JOIN #prom_mes t3
  ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.AÑO = {int(Year)}
  AND t1.MES IN ({int(Mes_imp1)},{int(Mes_imp2)},{int(Mes_imp3)},{int(Mes_imp4)},{int(Mes_imp5)},{int(Mes_imp6)},{int(Mes_imp7)},{int(Mes_imp8)},{int(Mes_imp9)},{int(Mes_imp10)},{int(Mes_imp11)},{int(Mes_imp12)})
  AND t1.MONEDA IN ('{Moneda}')
"""
work_conn.execute(text(sql_ejecucion_b_dip_aper_imp))


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip_aper_imp WHERE C_ENTIDAD IN (1500040001)"))
_log("DELETE #ejecucion_b_dip_aper_imp (entidad 1500040001)", res.rowcount)


In [ ]:
sql_delete_b = f"""
DELETE FROM #ejecucion_b_dip_aper_imp
WHERE C_ENTIDAD NOT IN ({int(PACAPR1)},{int(PACAPR2)},{int(PACAPR3)},{int(PACAPR4)},{int(PACAPR5)},{int(PACAPR6)},{int(PACAPR7)}) AND MONEDA IN ('{Moneda}')
"""
res = work_conn.execute(text(sql_delete_b))
_log("DELETE #ejecucion_b_dip_aper_imp (fuera PACAPR)", res.rowcount)
ejecucion_b_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_b_dip_aper_imp"), work_conn)
_log("ejecucion_b_dip_aper_imp", ejecucion_b_dip_aper_imp)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_aper_imp"))
sql_ejecucion_c_dip_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       T1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_c_dip_aper_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND T1.C_ENTIDAD = T2.ENTIDAD)
LEFT JOIN #prom_mes t3
  ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)}
  AND t1.MES IN ({int(Mes_imp1)},{int(Mes_imp2)},{int(Mes_imp3)},{int(Mes_imp4)},{int(Mes_imp5)},{int(Mes_imp6)},{int(Mes_imp7)},{int(Mes_imp8)},{int(Mes_imp9)},{int(Mes_imp10)},{int(Mes_imp11)},{int(Mes_imp12)})
  AND t1.MONEDA IN ('{Moneda}')
"""
work_conn.execute(text(sql_ejecucion_c_dip_aper_imp))


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip_aper_imp WHERE C_ENTIDAD IN (1500040001)"))
_log("DELETE #ejecucion_c_dip_aper_imp (entidad 1500040001)", res.rowcount)


In [ ]:
sql_delete_c = f"""
DELETE FROM #ejecucion_c_dip_aper_imp
WHERE C_ENTIDAD NOT IN ({int(PACAPR1)},{int(PACAPR2)},{int(PACAPR3)},{int(PACAPR4)},{int(PACAPR5)},{int(PACAPR6)},{int(PACAPR7)}) AND MONEDA IN ('{Moneda}')
"""
res = work_conn.execute(text(sql_delete_c))
_log("DELETE #ejecucion_c_dip_aper_imp (fuera PACAPR)", res.rowcount)
ejecucion_c_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_c_dip_aper_imp"), work_conn)
_log("ejecucion_c_dip_aper_imp", ejecucion_c_dip_aper_imp)


In [ ]:
# /*IMPUTACIONES:Transacciones diferenciadas para coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip_aper_imp"))
sql_ejecucion_d_dip_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       T1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_ACTIVO AS C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_d_dip_aper_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND T1.C_ENTIDAD = T2.ENTIDAD)
LEFT JOIN #prom_mes t3
  ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
WHERE t2.OBS IN ('CR18_difcoy') AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.C_SCN NOT IN ('TC') AND t1.AÑO = {int(Year)}
  AND t1.MES IN ({int(Mes_imp1)},{int(Mes_imp2)},{int(Mes_imp3)},{int(Mes_imp4)},{int(Mes_imp5)},{int(Mes_imp6)},{int(Mes_imp7)},{int(Mes_imp8)},{int(Mes_imp9)},{int(Mes_imp10)},{int(Mes_imp11)},{int(Mes_imp12)})
  AND t1.MONEDA IN ('{Moneda}')
"""
work_conn.execute(text(sql_ejecucion_d_dip_aper_imp))


In [ ]:
sql_delete_d = f"""
DELETE FROM #ejecucion_d_dip_aper_imp
WHERE C_ENTIDAD NOT IN ({int(PACAPR1)},{int(PACAPR2)},{int(PACAPR3)},{int(PACAPR4)},{int(PACAPR5)},{int(PACAPR6)},{int(PACAPR7)}) AND MONEDA IN ('{Moneda}')
"""
res = work_conn.execute(text(sql_delete_d))
_log("DELETE #ejecucion_d_dip_aper_imp (fuera PACAPR)", res.rowcount)
ejecucion_d_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_d_dip_aper_imp"), work_conn)
_log("ejecucion_d_dip_aper_imp", ejecucion_d_dip_aper_imp)


In [ ]:
# /*IMPUTACIONES:Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip_aper_imp"))
sql_ejecucion_f_dip_aper_imp = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       T1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       t2.C_CUP AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_f_dip_aper_imp
FROM #ejec_dip_y t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON (t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
      AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
      AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND T1.C_ENTIDAD = T2.ENTIDAD)
LEFT JOIN #prom_mes t3
  ON (t1.MONEDA = t3.MONEDA) AND (t1.MES = t3.MES) AND (t1.AÑO = t3.AÑO)
WHERE t2.OBS IN ('CR18_difact') AND t2.C_SCN NOT IN ('TC') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)}
  AND t1.MES IN ({int(Mes_imp1)},{int(Mes_imp2)},{int(Mes_imp3)},{int(Mes_imp4)},{int(Mes_imp5)},{int(Mes_imp6)},{int(Mes_imp7)},{int(Mes_imp8)},{int(Mes_imp9)},{int(Mes_imp10)},{int(Mes_imp11)},{int(Mes_imp12)})
  AND t1.MONEDA IN ('{Moneda}')
"""
work_conn.execute(text(sql_ejecucion_f_dip_aper_imp))


In [ ]:
sql_delete_f = f"""
DELETE FROM #ejecucion_f_dip_aper_imp
WHERE C_ENTIDAD NOT IN ({int(PACAPR1)},{int(PACAPR2)},{int(PACAPR3)},{int(PACAPR4)},{int(PACAPR5)},{int(PACAPR6)},{int(PACAPR7)}) AND MONEDA IN ('{Moneda}')
"""
res = work_conn.execute(text(sql_delete_f))
_log("DELETE #ejecucion_f_dip_aper_imp (fuera PACAPR)", res.rowcount)
ejecucion_f_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_f_dip_aper_imp"), work_conn)
_log("ejecucion_f_dip_aper_imp", ejecucion_f_dip_aper_imp)


In [ ]:
# EJECUCION_G_DIP_aper_imp (DROP=ACTIVIDAD)
# GROUP BY exactamente como en el SAS: no incluye ACTIVIDAD (solo se usa para
# derivar C_ACT en el SELECT); PROC SQL toma el valor del primer registro del
# grupo para esa columna al calcular C_ACT
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip_aper_imp"))
sql_ejecucion_g_dip_aper_imp = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       t1.TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_TRANSACCION,
       t1.T_TRANSACCION,
       t1.N_CUENTA,
       t1.C_ACTIVO,
       t1.C_CUP,
       (CASE WHEN MIN(t1.ACTIVIDAD) = 'FFAA' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'FFOO' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'PREVISION' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'FONASA' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'CORFO' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'CENABAST' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'AAPP' THEN 'AAPP'
             WHEN MIN(t1.ACTIVIDAD) = 'SSPP' THEN 'SSPP'
             WHEN MIN(t1.ACTIVIDAD) = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       t1.dato
INTO #ejecucion_g_dip_aper_imp
FROM #ejec_z_mixt_aper_dip_imp t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    t1.TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_SCN,
    t1.C_CAGENTE,
    t1.C_TRANSACCION,
    t1.T_TRANSACCION,
    t1.N_CUENTA,
    t1.C_ACTIVO,
    t1.C_CUP,
    t1.C_ASIGNACION,
    t1.N_ANALITICO,
    t1.NIVEL,
    t1.dato
"""
work_conn.execute(text(sql_ejecucion_g_dip_aper_imp))
ejecucion_g_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_g_dip_aper_imp"), work_conn)
_log("ejecucion_g_dip_aper_imp", ejecucion_g_dip_aper_imp)


In [ ]:
# /*IMPUTACIONES:UNE TABLAS de Salida total de ejecución CG coy de data DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_imp"))
sql_union = """
SELECT * INTO #ejecucion_cg_dip_aper_imp FROM (
  SELECT * FROM #ejecucion_a_dip_aper_imp
  UNION ALL
  SELECT * FROM #ejecucion_b_dip_aper_imp
  UNION ALL
  SELECT * FROM #ejecucion_c_dip_aper_imp
  UNION ALL
  SELECT * FROM #ejecucion_d_dip_aper_imp
  UNION ALL
  SELECT * FROM #ejecucion_g_dip_aper_imp
  UNION ALL
  SELECT * FROM #ejecucion_f_dip_aper_imp
) u
"""
work_conn.execute(text(sql_union))


In [ ]:
# DROP TABLE de las intermedias (equivalente server-side: se dropean las #tmp)
for _t in ["#ejecucion_a_prelim_dip_aper_imp", "#ejecucion_a_dip_aper_imp", "#ejecucion_b_dip_aper_imp", "#ejecucion_c_dip_aper_imp", "#ejecucion_d_dip_aper_imp", "#ejecucion_g_dip_aper_imp", "#ejecucion_f_dip_aper_imp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# UPDATE EJECUCION_CG_DIP_aper_imp SET N_CUENTA='Imputacion'
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_aper_imp SET N_CUENTA = 'Imputacion'"))
_log("UPDATE #ejecucion_cg_dip_aper_imp (N_CUENTA=Imputacion)", res.rowcount)
ejecucion_cg_dip_aper_imp = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_aper_imp"), work_conn)
_log("ejecucion_cg_dip_aper_imp", ejecucion_cg_dip_aper_imp)


## OPC Salida CG_coy (solo DIP)

Arma la ejecución de gasto/consumo de gobierno central de coyuntura (solo DIPRES) del mes t: analíticos predefinidos y variables, transferencias en especie de salud abiertas por ratio, ingresos operacionales imputados por entidades que informan agregado, e imputaciones permanentes incorporadas al total

*confianza: medium · verificador: approve · SAS: PROC SQL: múltiples CREATE TABLE con LEFT JOIN, UNION ALL, DELETE/UPDATE y PROC DATASETS APPEND, encadenados sobre tablas WORK*

In [ ]:
# ========= OPC Salida CG_coy (solo DIP) =========
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
# /*Crea tabla de apertura por actividad por PACACR ante la falta de C_ENTIDAD en EJEC_DIPRES */
# /*MODIFICADO JUNIO 2025*/ - /*Ejecucion dipres del periodo, con entidad asignada*/
sql_ejec_dip = f"""
SELECT T1.*, T2.C_ENTIDAD, T2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
    ON T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA
    AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.AÑO = {int(Year)} AND T1.Mes = {int(Mes)}
"""
work_conn.execute(text(sql_ejec_dip))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip"))
# /*Analiticos predefinidos */
sql_ejecucion_a_prelim_dip = """
SELECT t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1
          WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3
          ELSE 4 END) AS TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
    t2.C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t2.C_ACTIVO,
    t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST(t1.DEVENG * t3.DATO / 1000000 AS decimal(18,2)) AS DEVENGO,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_a_prelim_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip))


In [ ]:
# /*nuevo, para eliminar ingresos operacionales que vienen agregados en data dipres. Se incorporan al final del proceso*/
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE C_CUENTA = '07' AND C_ITEM = '0'"))
_log("DELETE #ejecucion_a_prelim_dip (07/0)", res.rowcount)


In [ ]:
# /*Estructura para abrir transferencias en especie...*/
# RATIOS ya existe en memoria (dependencia 4- Salida EJEC hasta CGR); se sube a #tmp para operar server-side
work_conn.execute(text("DROP TABLE IF EXISTS #ratios"))
ratios.to_sql("#ratios", work_conn, if_exists="replace", index=False)
work_conn.execute(text("DROP TABLE IF EXISTS #ratios_d63"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION,
    t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.Ratio
INTO #ratios_d63
FROM #ratios t1
WHERE t1.C_SCN = 'D63'
"""))


In [ ]:
# /*apertura transferencias en especie de la salud pública*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip2"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION,
    t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO, t1.[KEY],
    t2.Ratio
INTO #ejecucion_a_prelim_dip2
FROM #ejecucion_a_prelim_dip t1
LEFT JOIN #ratios_d63 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_CUENTA = t2.C_CUENTA AND t1.AÑO = t2.AÑO
"""))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip"))
work_conn.execute(text("EXEC sp_rename '#ejecucion_a_prelim_dip2', '#ejecucion_a_prelim_dip'"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_ajtesp"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_,
    'D63' AS C_SCN, 'S14' AS C_CAGENTE, '433' AS C_TRANSACCION, 'transfesp' AS T_TRANSACCION,
    'IG' AS N_CUENTA, t1.C_ACTIVO, '272' AS C_CUP, t1.C_ACT, t1.NIVEL,
    t1.DEVENGO * t1.Ratio AS DEVENGO, t1.[KEY]
INTO #ejecucion_a_prelim_dip_ajtesp
FROM #ejecucion_a_prelim_dip t1
WHERE t1.Ratio IS NOT NULL
"""))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip3"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION,
    t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION,
    t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
    (CASE WHEN t1.Ratio IS NOT NULL THEN t1.DEVENGO * (1 - t1.Ratio) ELSE t1.DEVENGO END) AS DEVENGO,
    t1.[KEY]
INTO #ejecucion_a_prelim_dip3
FROM #ejecucion_a_prelim_dip t1
"""))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip"))
work_conn.execute(text("EXEC sp_rename '#ejecucion_a_prelim_dip3', '#ejecucion_a_prelim_dip'"))


In [ ]:
# PROC DATASETS APPEND BASE=EJECUCION_A_prelim_DIP DATA=EJECUCION_A_prelim_DIP_ajtesp FORCE
cols_ejecucion_a_prelim_dip = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, "
    "C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, ANALITICO_, "
    "C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, "
    "DEVENGO, [KEY]"
)
sql_append = f"INSERT INTO #ejecucion_a_prelim_dip ({cols_ejecucion_a_prelim_dip}) SELECT {cols_ejecucion_a_prelim_dip} FROM #ejecucion_a_prelim_dip_ajtesp"
res = work_conn.execute(text(sql_append))
_log("APPEND #ejecucion_a_prelim_dip_ajtesp -> #ejecucion_a_prelim_dip", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE [KEY] = '15_04_01'"))
_log("DELETE #ejecucion_a_prelim_dip (15_04_01)", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_dip (32/analiticos)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text("""
DELETE FROM #ejecucion_a_prelim_dip
WHERE [KEY] = '07_06_01' AND C_CUENTA = '06' AND C_ITEM = '01'
    AND C_ASIGNACION IN ('000','00') AND C_ANALITICO IN ('01000000')
"""))
_log("DELETE #ejecucion_a_prelim_dip (07_06_01 CORFO)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos (sigue). GENERA BASE AGREGADA*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a_dip
FROM #ejecucion_a_prelim_dip t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""))


In [ ]:
# /*Analiticos variables */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t2.C_SCN, t2.C_CAGENTE,
    t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST(t1.DEVENG * t3.DATO / 1000000 AS decimal(18,2)) AS Dato,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_b_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t1.C_ANALITICO IN ('04001000','04002000') AND t1.C_ENTIDAD NOT IN (5601,10201)
"""))


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip WHERE [KEY] = '15_04_01'"))
_log("DELETE #ejecucion_b_dip (15_04_01)", res.rowcount)


In [ ]:
# /*ANALITICOS VARIABLES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t2.C_SCN, t2.C_CAGENTE,
    t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST(t1.DEVENG * t3.DATO / 1000000 AS decimal(18,2)) AS Dato,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_c_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
    AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t2.C_SCN NOT IN ('TC') AND t1.C_ENTIDAD NOT IN (5601,10201)
"""))


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip WHERE [KEY] = '15_04_01'"))
_log("DELETE #ejecucion_c_dip (15_04_01)", res.rowcount)


In [ ]:
# /* Transacciones diferenciadas para coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t2.C_ACTIVO AS C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA,
    t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST(t1.DEVENG * t3.DATO / 1000000 AS decimal(18,2)) AS Dato,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_d_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difcoy') AND t1.C_ENTIDAD NOT IN (5601,10201)
    AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t2.C_SCN NOT IN ('TC')
"""))


In [ ]:
# /* Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t2.C_SCN, t2.C_CAGENTE,
    t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t2.C_CUP AS C_ACT, t1.NIVEL,
    CAST(t1.DEVENG * t3.DATO / 1000000 AS decimal(18,2)) AS Dato,
    CONCAT_WS('_', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA) AS [KEY]
INTO #ejecucion_f_dip
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
    AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
    AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""))


In [ ]:
# EJECUCION_G_DIP (DROP=ACTIVIDAD) desde EJEC_G_MIXT_DIP, que ya está en memoria (dependencia 5- TCons Mixtas_M_DIP_MOD)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_dip"))
ejec_g_mixt_dip.to_sql("#ejec_g_mixt_dip", work_conn, if_exists="replace", index=False)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL, t1.dato,
    CONCAT_WS('_', t1.MONEDA) AS [KEY]
INTO #ejecucion_g_dip
FROM #ejec_g_mixt_dip t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL, t1.dato
"""))


In [ ]:
work_conn.execute(text("UPDATE #ejecucion_g_dip SET C_ACTIVO = 'TC'"))


In [ ]:
# /*UNE TABLAS de Salida total de ejecución CG coy de data DIPRES*/ (DROP=KEY se aplica excluyendo la columna del SELECT)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_prev"))
cols_union_dip = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, "
    "C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato"
)
sql_cg_dip_prev = f"""
SELECT {cols_union_dip} INTO #ejecucion_cg_dip_prev FROM #ejecucion_a_dip
UNION ALL SELECT {cols_union_dip} FROM #ejecucion_b_dip
UNION ALL SELECT {cols_union_dip} FROM #ejecucion_c_dip
UNION ALL SELECT {cols_union_dip} FROM #ejecucion_d_dip
UNION ALL SELECT {cols_union_dip} FROM #ejecucion_g_dip
UNION ALL SELECT {cols_union_dip} FROM #ejecucion_f_dip
"""
work_conn.execute(text(sql_cg_dip_prev))


In [ ]:
for t in ["#ejecucion_a_prelim_dip", "#ejecucion_a_dip", "#ejecucion_b_dip", "#ejecucion_c_dip", "#ejecucion_d_dip", "#ejecucion_g_dip", "#ejecucion_f_dip"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
work_conn.execute(text("UPDATE #ejecucion_cg_dip_prev SET N_CUENTA = 'DIPRES'"))


In [ ]:
# /*Incorporacion de imputaciones permanentes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imput_perm"))
sql_imput_perm = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP,
    (CASE WHEN t1.C_ACT = 'FFAA' THEN 'AAPP'
          WHEN t1.C_ACT = 'FFOO' THEN 'AAPP'
          WHEN t1.C_ACT = 'PREVISION' THEN 'AAPP'
          WHEN t1.C_ACT = 'FONASA' THEN 'AAPP'
          WHEN t1.C_ACT = 'CORFO' THEN 'AAPP'
          WHEN t1.C_ACT = 'CENABAST' THEN 'AAPP'
          WHEN t1.C_ACT = 'AAPP' THEN 'AAPP'
          WHEN t1.C_ACT = 'SSPP' THEN 'SSPP'
          WHEN t1.C_ACT = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL, t1.Dato
INTO #ejecucion_cg_dip_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP t1
WHERE t1.AÑO = {int(Year)}
    AND t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_imput_perm))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip"))
work_conn.execute(text("""
SELECT * INTO #ejecucion_cg_dip FROM #ejecucion_cg_dip_prev
UNION ALL
SELECT * FROM #ejecucion_cg_dip_imput_perm
"""))
ejecucion_cg_dip_imput_perm = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_imput_perm"), work_conn)
_log("ejecucion_cg_dip_imput_perm", ejecucion_cg_dip_imput_perm)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imput_perm"))


In [ ]:
# /*NUEVO: IMPUTA INGRESOS OPERACIONALES PARA LAS ENTIDADES QUE LO INFORMAN AGREGADO EN DIPRES, ASÍ SE MEJORA EL INDICADOR DE VENTAS*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_07"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2
          WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA,
    t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
          WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
          WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
    t1.NIVEL,
    CAST(t1.DEVENG * t3.DATO / 1000000 AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_dip_07
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('07') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""))


In [ ]:
# /*Verifica que entidades informan solo el agregado para filtrarlas*/
# COUNT()=1 filtra entidades con un único registro por esta llave; nota del analista: sin
# validación previa de duplicados, un cambio de granularidad en EJEC_DIPRES altera el filtro
work_conn.execute(text("DROP TABLE IF EXISTS #cuenta_item_07"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP,
    COUNT(t1.DEVENGO) AS id
INTO #cuenta_item_07
FROM #ejecucion_a_dip_07 t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA,
    t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP
"""))
cuenta_item_07 = pd.read_sql(text("SELECT * FROM #cuenta_item_07"), work_conn)
_log("cuenta_item_07", cuenta_item_07)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_dip_07"))
work_conn.execute(text("""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
    'DIPRES' AS N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO AS Dato
INTO #ejecucion_dip_07
FROM #ejecucion_a_dip_07 t1
LEFT JOIN #cuenta_item_07 t2
    ON t1.MONEDA = t2.MONEDA AND t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.C_PARTIDA = t2.C_PARTIDA
    AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
    AND t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
WHERE t2.id = 1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.C_ACTIVO,
    t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO
"""))


In [ ]:
# DATA EJECUCION_CG_DIP; SET EJECUCION_CG_DIP EJECUCION_DIP_07; -> concatenación server-side (append)
cols_cg_dip = (
    "SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, "
    "C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato"
)
sql_append_cg_dip = f"INSERT INTO #ejecucion_cg_dip ({cols_cg_dip}) SELECT {cols_cg_dip} FROM #ejecucion_dip_07"
res = work_conn.execute(text(sql_append_cg_dip))
_log("APPEND #ejecucion_dip_07 -> #ejecucion_cg_dip", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_cg_dip WHERE NIVEL = 'EXCLUIDO'"))
_log("DELETE #ejecucion_cg_dip (EXCLUIDO)", res.rowcount)


In [ ]:
for t in ["#ejecucion_a_dip_07", "#ejecucion_dip_07", "#ejec_dip", "#cuenta_item_07", "#ejecucion_a_prelim_dip_ajtesp"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
# Lecturas finales de control para dejar los datasets de salida en memoria, igual que el resto del notebook
ejecucion_cg_dip = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip"), work_conn)
_log("ejecucion_cg_dip", ejecucion_cg_dip)


## OPC Aper_CG_coy (solo DIP)

Ejecución desagregada de gasto de gobierno central (coyuntura) solo con datos DIPRES del mes vigente: clasifica analíticos SCN, abre transferencias en especie de salud pública según ratio, excluye entidades y analíticos ajustados por definición metodológica, agrega imputaciones permanentes e imputa ingresos operacionales de entidades que informan agregado

*confianza: low · verificador: revise · SAS: PROC SQL: múltiples CREATE TABLE con LEFT JOIN, CASE WHEN, agregaciones, UNION ALL, DELETE, UPDATE y APPEND encadenados sobre datasets WORK derivados de tablas GOBGENER*

In [ ]:
# ========= OPC Aper_CG_coy (solo DIP) =========
# /*MODIFICADO JUNIO 2025*/ Ejecucion dipres del periodo, con entidad asignada
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip"))
sql_ejec_dip = f"""
SELECT T1.*, T2.C_ENTIDAD, T2.ACTIVIDAD, T2.NIVEL
INTO #ejec_dip
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
  ON T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
WHERE T1.AÑO = {int(Year)} AND T1.MES = {int(Mes)}
"""
work_conn.execute(text(sql_ejec_dip))
ejec_dip = pd.read_sql(text("SELECT * FROM #ejec_dip"), work_conn)
_log("ejec_dip", ejec_dip)


In [ ]:
# Analiticos predefinidos
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
sql_ejecucion_a_prelim_dip_aper = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
  ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip_aper))


In [ ]:
# nuevo, para eliminar ingresos operacionales que vienen agregados en data dipres. Se incorporan al final del proceso
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA = '07' AND C_ITEM = '0'"))
_log("DELETE #ejecucion_a_prelim_dip_aper (07/0)", res.rowcount)


In [ ]:
# Estructura para abrir transferencias en especie... (RATIOS ya viene de un nodo anterior, dataframe en memoria)
ratios_d63 = ratios[ratios["C_SCN"] == "D63"][
    ["SUBSECTOR", "AÑO", "MONEDA", "C_ENTIDAD", "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA",
     "C_TIPO_CUENTA", "C_CUENTA", "C_SCN", "C_CAGENTE", "C_TRANSACCION", "T_TRANSACCION",
     "N_CUENTA", "C_ACTIVO", "C_CUP", "C_ACT", "NIVEL", "Ratio"]
].copy()
ratios_d63.to_sql("#ratios_d63", work_conn, if_exists="replace", index=False)
_log("ratios_d63", ratios_d63)


In [ ]:
# apertura transferencias en especie de la salud pública
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper2"))
sql_ejecucion_a_prelim_dip_aper2 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
       t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO, t2.Ratio
INTO #ejecucion_a_prelim_dip_aper2
FROM #ejecucion_a_prelim_dip_aper t1
LEFT JOIN #ratios_d63 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
 AND t1.C_CUENTA = t2.C_CUENTA AND t1.AÑO = t2.AÑO
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip_aper2))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
work_conn.execute(text("EXEC sp_rename '#ejecucion_a_prelim_dip_aper2', '#ejecucion_a_prelim_dip_aper'"))


In [ ]:
# EJECUCION_A_prelim_DIP_ajtesp_ap: apertura fija D63/S14/433/transfesp/IG/272 cuando Ratio no es nulo
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_ajtesp_ap"))
sql_ajtesp_ap = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.ANALITICO_,
       'D63' AS C_SCN, 'S14' AS C_CAGENTE, '433' AS C_TRANSACCION, 'transfesp' AS T_TRANSACCION,
       'IG' AS N_CUENTA, t1.C_ACTIVO, '272' AS C_CUP, t1.C_ACT, t1.NIVEL,
       t1.DEVENGO * t1.Ratio AS DEVENGO
INTO #ejecucion_a_prelim_dip_ajtesp_ap
FROM #ejecucion_a_prelim_dip_aper t1
WHERE t1.Ratio IS NOT NULL
"""
work_conn.execute(text(sql_ajtesp_ap))
# riesgo señalado por el analista: SAS no hace DROP de columnas temporales (ANALITICO_, C_ACT) en esta tabla; se conserva igual
ejecucion_a_prelim_dip_ajtesp_ap = pd.read_sql(text("SELECT * FROM #ejecucion_a_prelim_dip_ajtesp_ap"), work_conn)
_log("ejecucion_a_prelim_dip_ajtesp_ap", ejecucion_a_prelim_dip_ajtesp_ap)


In [ ]:
# recalcula DEVENGO aplicando (1-Ratio) donde Ratio no es nulo, deja DEVENGO igual si Ratio es nulo
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper2"))
sql_recalculo = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.ANALITICO_, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
       t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       (CASE WHEN t1.Ratio IS NOT NULL THEN t1.DEVENGO * (1 - t1.Ratio) ELSE t1.DEVENGO END) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper2
FROM #ejecucion_a_prelim_dip_aper t1
"""
work_conn.execute(text(sql_recalculo))
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
work_conn.execute(text("EXEC sp_rename '#ejecucion_a_prelim_dip_aper2', '#ejecucion_a_prelim_dip_aper'"))


In [ ]:
# PROC DATASETS APPEND: agrega la apertura de transf. en especie a la tabla base, server-side
cols_ajtesp_ap = ", ".join([
    "SUBSECTOR", "MONEDA", "AÑO", "TRIM", "MES", "C_ENTIDAD", "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA",
    "C_TIPO_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "N_ANALITICO", "ANALITICO_",
    "C_SCN", "C_CAGENTE", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_ACTIVO", "C_CUP", "C_ACT",
    "NIVEL", "DEVENGO",
])
sql_append_ajtesp = f"INSERT INTO #ejecucion_a_prelim_dip_aper ({cols_ajtesp_ap}) SELECT {cols_ajtesp_ap} FROM #ejecucion_a_prelim_dip_ajtesp_ap"
res = work_conn.execute(text(sql_append_ajtesp))
_log("APPEND #ejecucion_a_prelim_dip_aper (ajtesp_ap)", res.rowcount)
# fin apertura de transf en especie de salud pública


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_a_prelim_dip_aper (10201)", res.rowcount)


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_dip_aper (32/analiticos)", res.rowcount)


In [ ]:
# Analiticos predefinidos: AJUSTE COYUNTURA - se elimina indicador transacción CORFO que se desagrega anualmente entre ventas y D45
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_ENTIDAD IN (3801) AND C_CUENTA = '06' AND C_ITEM = '01' AND C_ASIGNACION IN ('00','000') AND C_ANALITICO IN ('01000000')"))
_log("DELETE #ejecucion_a_prelim_dip_aper (3801 CORFO)", res.rowcount)


In [ ]:
# Analiticos predefinidos (sigue). BASE DESAGREGADA -- agrega por combinación de columnas del GROUP BY
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_aper"))
sql_ejecucion_a_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a_dip_aper
FROM #ejecucion_a_prelim_dip_aper t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
         t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
         t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
         t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a_dip_aper))


In [ ]:
# Analiticos variables
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip_aper"))
sql_ejecucion_b_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_b_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
 AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
 AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ANALITICO IN ('04001000','04002000') AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_b_dip_aper))


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_b_dip_aper (10201)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_aper"))
sql_ejecucion_c_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_c_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
 AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
 AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
"""
work_conn.execute(text(sql_ejecucion_c_dip_aper))


In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip_aper WHERE C_ENTIDAD IN (10201)"))
_log("DELETE #ejecucion_c_dip_aper (10201)", res.rowcount)


In [ ]:
# Transacciones diferenciadas para coyuntura
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip_aper"))
sql_ejecucion_d_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_ACTIVO AS C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_d_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
 AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
 AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difcoy') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.C_SCN NOT IN ('TC')
"""
work_conn.execute(text(sql_ejecucion_d_dip_aper))


In [ ]:
# Transacciones diferenciadas por actividad
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip_aper"))
sql_ejecucion_f_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       t2.C_CUP AS C_ACT, t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS Dato
INTO #ejecucion_f_dip_aper
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
  ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
 AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
 AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS IN ('CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_ejecucion_f_dip_aper))


In [ ]:
# EJECUCION_G_DIP_APER (DROP=ACTIVIDAD) -- se usa el ejec_g_mixt_aper_dip que dejó el nodo de TCons Mixtas_M_DIP_MOD
ejec_g_mixt_aper_dip.to_sql("#ejec_g_mixt_aper_dip", work_conn, if_exists="replace", index=False)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip_aper"))
sql_ejecucion_g_dip_aper = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       SUM(t1.dato) AS Dato
INTO #ejecucion_g_dip_aper
FROM #ejec_g_mixt_aper_dip t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
         t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION,
         t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_g_dip_aper))


In [ ]:
# UNE TABLAS de Salida total de ejecución CG coy de data DIPRES -- UNION ALL sin ORDER BY: orden no determinista (riesgo señalado por el analista)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_prev"))
sql_union = """
SELECT * INTO #ejecucion_cg_dip_aper_prev FROM #ejecucion_a_dip_aper
UNION ALL
SELECT * FROM #ejecucion_b_dip_aper
UNION ALL
SELECT * FROM #ejecucion_c_dip_aper
UNION ALL
SELECT * FROM #ejecucion_d_dip_aper
UNION ALL
SELECT * FROM #ejecucion_g_dip_aper
UNION ALL
SELECT * FROM #ejecucion_f_dip_aper
"""
work_conn.execute(text(sql_union))
for _t in ["#ejecucion_a_prelim_dip_aper", "#ejecucion_a_dip_aper", "#ejecucion_b_dip_aper", "#ejecucion_c_dip_aper", "#ejecucion_d_dip_aper", "#ejecucion_g_dip_aper", "#ejecucion_f_dip_aper"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_aper_prev SET N_CUENTA = 'DIPRES'"))
_log("UPDATE #ejecucion_cg_dip_aper_prev (N_CUENTA=DIPRES)", res.rowcount)
# Se materializa: WORK.EJECUCION_CG_DIP_APER_PREV es salida del nodo
ejecucion_cg_dip_aper_prev = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_aper_prev"), work_conn)
_log("ejecucion_cg_dip_aper_prev", ejecucion_cg_dip_aper_prev)


In [ ]:
# Incorporacion de imputaciones permanentes -- GOBGENER.IMPUT_GC_DIP_APER es tabla de BD (la deja el nodo OPC Agregar IMPUT a permanentes)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cg_dip_aper_imput_perm"))
sql_imput_perm = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
       t1.C_ACTIVO, t1.C_CUP,
       (CASE WHEN t1.C_ACT = 'FFAA' THEN 'AAPP'
             WHEN t1.C_ACT = 'FFOO' THEN 'AAPP'
             WHEN t1.C_ACT = 'PREVISION' THEN 'AAPP'
             WHEN t1.C_ACT = 'FONASA' THEN 'AAPP'
             WHEN t1.C_ACT = 'CORFO' THEN 'AAPP'
             WHEN t1.C_ACT = 'CENABAST' THEN 'AAPP'
             WHEN t1.C_ACT = 'AAPP' THEN 'AAPP'
             WHEN t1.C_ACT = 'SSPP' THEN 'SSPP'
             WHEN t1.C_ACT = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       t1.Dato
INTO #ejec_cg_dip_aper_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER t1
WHERE t1.AÑO = {int(Year)} AND t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_imput_perm))
ejec_cg_dip_aper_imput_perm = pd.read_sql(text("SELECT * FROM #ejec_cg_dip_aper_imput_perm"), work_conn)
_log("ejec_cg_dip_aper_imput_perm", ejec_cg_dip_aper_imput_perm)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper"))
sql_ejecucion_cg_dip_aper = """
SELECT * INTO #ejecucion_cg_dip_aper FROM #ejecucion_cg_dip_aper_prev
UNION ALL
SELECT * FROM #ejec_cg_dip_aper_imput_perm
"""
work_conn.execute(text(sql_ejecucion_cg_dip_aper))


In [ ]:
# NUEVO: IMPUTA INGRESOS OPERACIONALES PARA LAS ENTIDADES QUE LO INFORMAN AGREGADO EN DIPRES, ASÍ SE MEJORA EL INDICADOR DE VENTAS
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_07"))
sql_ejecucion_a_dip_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END) AS TRIM,
       t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
       t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       (CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
       t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
       (CASE WHEN t1.ACTIVIDAD = 'FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       CAST((t1.DEVENG * t3.DATO / 1000000) AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_dip_07
FROM #ejec_dip t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
  ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('07') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_ejecucion_a_dip_07))
ejecucion_a_dip_07 = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip_07"), work_conn)
_log("ejecucion_a_dip_07", ejecucion_a_dip_07)


In [ ]:
# Verifica que entidades informan solo el agregado para filtrarlas
work_conn.execute(text("DROP TABLE IF EXISTS #cuenta_item_07"))
sql_cuenta_item_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP,
       COUNT(t1.DEVENGO) AS id
INTO #cuenta_item_07
FROM #ejecucion_a_dip_07 t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
         t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP
"""
work_conn.execute(text(sql_cuenta_item_07))
cuenta_item_07 = pd.read_sql(text("SELECT * FROM #cuenta_item_07"), work_conn)
_log("cuenta_item_07", cuenta_item_07)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_dip_07"))
sql_ejecucion_dip_07 = """
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
       t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
       t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
       'DIPRES' AS N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
       t1.DEVENGO AS Dato
INTO #ejecucion_dip_07
FROM #ejecucion_a_dip_07 t1
LEFT JOIN #cuenta_item_07 t2
  ON t1.MONEDA = t2.MONEDA AND t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.C_PARTIDA = t2.C_PARTIDA
 AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA AND t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA
 AND t1.C_CUENTA = t2.C_CUENTA
WHERE t2.id = 1
"""
work_conn.execute(text(sql_ejecucion_dip_07))
ejecucion_dip_07 = pd.read_sql(text("SELECT * FROM #ejecucion_dip_07"), work_conn)
_log("ejecucion_dip_07", ejecucion_dip_07)


In [ ]:
# data EJECUCION_CG_DIP_APER; set EJECUCION_CG_DIP_APER EJECUCION_DIP_07; -- una imputacion de ingresos operacionales
cols_cg_dip_aper = ", ".join([
    "SUBSECTOR", "MONEDA", "AÑO", "TRIM", "MES", "C_ENTIDAD", "C_PARTIDA", "C_CAPITULO", "C_PROGRAMA",
    "C_TIPO_CUENTA", "C_CUENTA", "C_ITEM", "C_ASIGNACION", "C_ANALITICO", "N_ANALITICO", "C_SCN",
    "C_CAGENTE", "C_TRANSACCION", "T_TRANSACCION", "N_CUENTA", "C_ACTIVO", "C_CUP", "C_ACT", "NIVEL", "Dato",
])
sql_append_dip_07 = f"INSERT INTO #ejecucion_cg_dip_aper ({cols_cg_dip_aper}) SELECT {cols_cg_dip_aper} FROM #ejecucion_dip_07"
res = work_conn.execute(text(sql_append_dip_07))
_log("APPEND #ejecucion_cg_dip_aper (dip_07)", res.rowcount)
ejecucion_cg_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_aper"), work_conn)
_log("ejecucion_cg_dip_aper", ejecucion_cg_dip_aper)


In [ ]:
# limpieza final de tablas temporales de sesión, equivalente al PROC SQL DROP TABLE del SAS
for _t in ["#ejec_cg_dip_aper_imput_perm", "#cuenta_item_07", "#ejecucion_a_dip_07", "#ejecucion_a_prelim_dip_ajtesp_ap", "#ejec_dip", "#ejecucion_dip_07", "#ejecucion_cg_dip_aper_prev", "#ejecucion_cg_dip_aper", "#ratios_d63"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## OPC Salida Ent CG_coy DIP

Ejecución de gasto de gobierno central de consumo (coyuntura), fuente DIPRES, del mes t, desagregada por analíticos predefinidos, variables, transacciones diferenciadas por coyuntura/actividad y mixtas, filtrada por el paquete de aperturas PACAPR y con imputaciones permanentes incorporadas

*confianza: medium · verificador: revise · SAS: PROC SQL: múltiples CREATE TABLE con LEFT JOIN, CASE WHEN, GROUP BY, UNION ALL y DELETE/UPDATE sobre tablas de trabajo temporales*

In [ ]:
# ========= OPC Salida Ent CG_coy DIP =========
work_conn.execute(text("DROP TABLE IF EXISTS #cr18_apertura_pacapr_v"))
# /*Crea tabla de apertura por actividad por PACACR ante la falta de C_ENTIDAD en EJEC_DIPRES */
sql_cr18_apertura_pacapr_v = """
SELECT DISTINCT t1.ACTIVIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.NIVEL
INTO #cr18_apertura_pacapr_v
FROM GOBGENER.dbo.CR18_ENTID_DIP t1
"""
work_conn.execute(text(sql_cr18_apertura_pacapr_v))
cr18_apertura_pacapr_v = pd.read_sql(text("SELECT * FROM #cr18_apertura_pacapr_v"), work_conn)
_log("cr18_apertura_pacapr_v", cr18_apertura_pacapr_v)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip"))
# /*Analiticos predefinidos */
# CATX de SAS omite componentes nulos/en blanco sin agregar separador de más;
# CONCAT_WS de SQL Server solo omite NULL, no '' -- se normaliza a NULL con
# NULLIF para que el KEY resultante coincida con el de SAS y no altere el
# filtro KEY IN (&PACAPR1..&PACAPR7)
sql_ejecucion_a_prelim_dip = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       (CASE WHEN t2.ANALITICO='M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END) AS ANALITICO_,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS DEVENGO,
       CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) AS KEY
INTO #ejecucion_a_prelim_dip
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2 ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr_v t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)} AND t1.MONEDA = '{Moneda}'
  AND CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip))


In [ ]:
# proc sql; delete from EJECUCION_A_prelim_DIP where c_cuenta='07' and c_item='0'; quit; /*nuevo*/
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE C_CUENTA = '07' AND C_ITEM = '0'"))
_log("DELETE #ejecucion_a_prelim_dip (c_cuenta=07,c_item=0)", res.rowcount)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE KEY = '15_04_01'"))
_log("DELETE #ejecucion_a_prelim_dip (KEY=15_04_01)", res.rowcount)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_dip (c_cuenta=32)", res.rowcount)
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip WHERE KEY = '07_06_01' AND C_CUENTA = '06' AND C_ITEM = '01' AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')"))
_log("DELETE #ejecucion_a_prelim_dip (KEY=07_06_01 ajuste CORFO)", res.rowcount)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip"))
# /*Analiticos predefinidos (sigue)*/
sql_ejecucion_a_dip = """
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       t1.TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_TRANSACCION,
       t1.T_TRANSACCION,
       t1.N_CUENTA,
       t1.C_ACTIVO,
       t1.C_CUP,
       t1.C_ACT,
       t1.NIVEL,
       SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a_dip
FROM #ejecucion_a_prelim_dip t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a_dip))
ejecucion_a_dip = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip"), work_conn)
_log("ejecucion_a_dip", ejecucion_a_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip"))
# /*Analiticos variables */
sql_ejecucion_b_dip = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato,
       CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) AS KEY
INTO #ejecucion_b_dip
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2 ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr_v t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)} AND t1.MONEDA = '{Moneda}'
  AND CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_b_dip))
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip WHERE KEY = '15_04_01'"))
_log("DELETE #ejecucion_b_dip (KEY=15_04_01)", res.rowcount)
ejecucion_b_dip = pd.read_sql(text("SELECT * FROM #ejecucion_b_dip"), work_conn)
_log("ejecucion_b_dip", ejecucion_b_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip"))
sql_ejecucion_c_dip = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato,
       CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) AS KEY
INTO #ejecucion_c_dip
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr_v t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)} AND t1.MONEDA = '{Moneda}'
  AND CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_c_dip))
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip WHERE KEY = '15_04_01'"))
_log("DELETE #ejecucion_c_dip (KEY=15_04_01)", res.rowcount)
ejecucion_c_dip = pd.read_sql(text("SELECT * FROM #ejecucion_c_dip"), work_conn)
_log("ejecucion_c_dip", ejecucion_c_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip"))
# /* Transacciones diferenciadas para coyuntura */
sql_ejecucion_d_dip = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t2.C_ACTIVO AS C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       (CASE WHEN t4.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t4.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t4.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato,
       CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) AS KEY
INTO #ejecucion_d_dip
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr_v t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t2.OBS IN ('CR18_difcoy') AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.C_SCN NOT IN ('TC') AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)} AND t1.MONEDA = '{Moneda}'
  AND CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_d_dip))
ejecucion_d_dip = pd.read_sql(text("SELECT * FROM #ejecucion_d_dip"), work_conn)
_log("ejecucion_d_dip", ejecucion_d_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip"))
# /* Transacciones diferenciadas por actividad */
sql_ejecucion_f_dip = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       (CASE WHEN t1.MES IN (1,2,3) THEN 1
             WHEN t1.MES IN (4,5,6) THEN 2
             WHEN t1.MES IN (7,8,9) THEN 3
             ELSE 4 END) AS TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       t2.C_CUP AS C_ACT,
       t4.NIVEL,
       (t1.DEVENG * t3.DATO / 1000000) AS Dato,
       CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) AS KEY
INTO #ejecucion_f_dip
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr_v t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t2.OBS IN ('CR18_difact') AND t2.C_SCN NOT IN ('TC') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)} AND t1.MONEDA = '{Moneda}'
  AND CONCAT_WS('_', NULLIF(t1.C_PARTIDA,''), NULLIF(t1.C_CAPITULO,''), NULLIF(t1.C_PROGRAMA,'')) IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_f_dip))
ejecucion_f_dip = pd.read_sql(text("SELECT * FROM #ejecucion_f_dip"), work_conn)
_log("ejecucion_f_dip", ejecucion_f_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip"))
# CREATE TABLE EJECUCION_G_DIP (DROP=ACTIVIDAD) -- KEY acá es CATX de un solo
# componente (MONEDA), que siempre viene poblado: CONCAT_WS('_', MONEDA) es
# equivalente sin riesgo de blancos
sql_ejecucion_g_dip = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       t1.TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_TRANSACCION,
       t1.T_TRANSACCION,
       t1.N_CUENTA,
       t1.C_ACTIVO,
       t1.C_CUP,
       (CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
             WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
             WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       t1.DATO,
       CONCAT_WS('_', t1.MONEDA) AS KEY
INTO #ejecucion_g_dip
FROM #ejec_g_mixt_dip t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t1.MONEDA = '{Moneda}'
  AND CONCAT_WS('_', t1.MONEDA) IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
         t1.ACTIVIDAD, t1.NIVEL, t1.DATO
"""
work_conn.execute(text(sql_ejecucion_g_dip))
res = work_conn.execute(text("UPDATE #ejecucion_g_dip SET C_ACTIVO = 'TC'"))
_log("UPDATE #ejecucion_g_dip C_ACTIVO=TC", res.rowcount)
ejecucion_g_dip = pd.read_sql(text("SELECT * FROM #ejecucion_g_dip"), work_conn)
_log("ejecucion_g_dip", ejecucion_g_dip)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_prev"))
# /*UNE TABLAS de Salida total de ejecución CG coy de data DIPRES*/
# CREATE TABLE ... (DROP=KEY): KEY era solo auxiliar de filtro, no se incluye
sql_ejecucion_cg_dip_prev = """
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato
INTO #ejecucion_cg_dip_prev
FROM #ejecucion_a_dip
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato
FROM #ejecucion_b_dip
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato
FROM #ejecucion_c_dip
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato
FROM #ejecucion_d_dip
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato
FROM #ejecucion_g_dip
UNION ALL
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE,
       C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato
FROM #ejecucion_f_dip
"""
work_conn.execute(text(sql_ejecucion_cg_dip_prev))
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_prev SET N_CUENTA = 'DIPRES'"))
_log("UPDATE #ejecucion_cg_dip_prev N_CUENTA=DIPRES", res.rowcount)


In [ ]:
# PROC SQL; DROP TABLE ... QUIT; -- limpieza de tablas intermedias del SAS;
# se replica dropeando las #tmp correspondientes (CR18_APERTURA_PACAPR_v
# quedaba comentado en el DROP original: se conserva viva, la usan bloques
# posteriores del notebook)
for _t in ["#ejecucion_a_prelim_dip", "#ejecucion_a_dip", "#ejecucion_b_dip", "#ejecucion_c_dip", "#ejecucion_d_dip", "#ejecucion_g_dip", "#ejecucion_f_dip"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imput_perm"))
# /*Incorporacion de imputaciones permanentes*/
# GOBGENER.IMPUT_GC_DIP.KEY ya viene precalculado en la tabla de BD (no se
# recalcula acá, es columna existente)
sql_ejecucion_cg_dip_imput_perm = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       t1.TRIM,
       t1.MES,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_SCN,
       t1.C_CAGENTE,
       t1.C_TRANSACCION,
       t1.T_TRANSACCION,
       t1.N_CUENTA,
       t1.C_ACTIVO,
       t1.C_CUP,
       (CASE WHEN t1.C_ACT='FFAA' THEN 'AAPP'
             WHEN t1.C_ACT='FFOO' THEN 'AAPP'
             WHEN t1.C_ACT='PREVISION' THEN 'AAPP'
             WHEN t1.C_ACT='FONASA' THEN 'AAPP'
             WHEN t1.C_ACT='CORFO' THEN 'AAPP'
             WHEN t1.C_ACT='CENABAST' THEN 'AAPP'
             WHEN t1.C_ACT='AAPP' THEN 'AAPP'
             WHEN t1.C_ACT='SSPP' THEN 'SSPP'
             WHEN t1.C_ACT='EEPP' THEN 'EEPP' END) AS C_ACT,
       t1.NIVEL,
       t1.Dato
INTO #ejecucion_cg_dip_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP t1
WHERE t1.AÑO = {int(Year)}
  AND t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.MONEDA = '{Moneda}'
  AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_cg_dip_imput_perm))
ejecucion_cg_dip_imput_perm = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_imput_perm"), work_conn)
_log("ejecucion_cg_dip_imput_perm", ejecucion_cg_dip_imput_perm)


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip"))
sql_ejecucion_cg_dip = """
SELECT * INTO #ejecucion_cg_dip FROM #ejecucion_cg_dip_prev
UNION ALL
SELECT * FROM #ejecucion_cg_dip_imput_perm
"""
work_conn.execute(text(sql_ejecucion_cg_dip))
ejecucion_cg_dip = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip"), work_conn)
_log("ejecucion_cg_dip", ejecucion_cg_dip)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imput_perm"))


## OPC Aper Ent CG_coy DIP

Ejecución coyuntural de Gobierno Central (CGR+DIPRES) desagregada por entidad y moneda específica, con analíticos predefinidos depurados, transacciones diferenciadas por coyuntura/actividad y las imputaciones permanentes incorporadas al cierre del periodo

*confianza: medium · verificador: approve · SAS: PROC SQL: múltiples CREATE TABLE con JOIN/CASE/GROUP BY, DELETE de analíticos predefinidos, UNION ALL de bloques y UPDATE, todo sobre tablas temporales de sesión*

In [ ]:
# ========= OPC Aper Ent CG_coy DIP =========
# Analíticos predefinidos: apertura por ACTIVIDAD/PARTIDA/CAPITULO/PROGRAMA a nivel DIP (bloque comentado del SAS ya no se usa)
work_conn.execute(text("DROP TABLE IF EXISTS #cr18_apertura_pacapr"))
sql_cr18_apertura_pacapr = """
SELECT DISTINCT t1.ACTIVIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.NIVEL
INTO #cr18_apertura_pacapr
FROM GOBGENER.dbo.CR18_ENTID_DIP t1
"""
work_conn.execute(text(sql_cr18_apertura_pacapr))
cr18_apertura_pacapr = pd.read_sql(text("SELECT * FROM #cr18_apertura_pacapr"), work_conn)
_log("cr18_apertura_pacapr", cr18_apertura_pacapr)


In [ ]:
# Analíticos predefinidos: EJECUCION_A_prelim_DIP_aper -> filtra cuentas fijas, cruza clasificación SCN_1, promedio mensual y apertura
# C_ENTIDAD se arma con INPUT(COMPRESS(CATX('0',partida,capitulo,programa), "1234567890.E-", "KIS"), E32.):
# COMPRESS con modificadores K/I/S invierte la lista (deja SOLO esos caracteres) e ignora mayúsc/minúsc y espacios;
# es decir, se queda con los dígitos/puntos/E/- de la concatenación y castea a número
cols_ej_a_prelim = (
    "SUBSECTOR, MONEDA, AÑO, "
    "CASE WHEN MES IN (1,2,3) THEN 1 WHEN MES IN (4,5,6) THEN 2 WHEN MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM, "
    "MES, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO"
)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
sql_ejecucion_a_prelim_dip_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
       t1.MES,
       TRY_CAST(REPLACE(REPLACE(REPLACE(CONCAT('0', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', ''), CHAR(9), ''), CHAR(13), '') AS float) AS C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END AS ANALITICO_,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       CASE WHEN t4.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t4.NIVEL,
       CAST(t1.DEVENG * t3.DATO / 1000000.0 AS decimal(18,2)) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2 ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_CUENTA NOT IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)}
  AND t1.MONEDA = '{Moneda}' AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_a_prelim_dip_aper))


In [ ]:
# proc sql; delete ... where c_cuenta='07' and c_item='0'; /*nuevo*/
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA = '07' AND C_ITEM = '0'"))
_log("DELETE #ejecucion_a_prelim_dip_aper (c_cuenta=07,c_item=0)", res.rowcount)


In [ ]:
# DELETE ... WHERE C_ENTIDAD IN (15004001)
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_ENTIDAD IN (15004001)"))
_log("DELETE #ejecucion_a_prelim_dip_aper (C_ENTIDAD 15004001)", res.rowcount)


In [ ]:
# DELETE ... WHERE C_CUENTA='32' AND C_ANALITICO IN ('04001000','04002000')
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_dip_aper (C_CUENTA=32)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper "
    "WHERE C_ENTIDAD IN (7006001) AND C_CUENTA = '06' AND C_ITEM = '01' AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')"
))
_log("DELETE #ejecucion_a_prelim_dip_aper (CORFO)", res.rowcount)


In [ ]:
# Analíticos predefinidos (sigue): EJECUCION_A_DIP_aper -> agrega DEVENGO por la clave de agrupación
# (SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL)
# el resto de columnas del SELECT (C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_ASIGNACION, C_ANALITICO, N_ANALITICO)
# no son función del GROUP BY: PROC SQL toma el valor del primer registro del grupo -> se replica con .first()
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_aper"))
sql_ejecucion_a_dip_aper = """
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES,
       MIN(C_ENTIDAD) AS C_ENTIDAD,
       MIN(C_PARTIDA) AS C_PARTIDA,
       MIN(C_CAPITULO) AS C_CAPITULO,
       MIN(C_PROGRAMA) AS C_PROGRAMA,
       C_TIPO_CUENTA, C_CUENTA, C_ITEM,
       MIN(C_ASIGNACION) AS C_ASIGNACION,
       MIN(C_ANALITICO) AS C_ANALITICO,
       MIN(N_ANALITICO) AS N_ANALITICO,
       C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL,
       SUM(DEVENGO) AS Dato
INTO #ejecucion_a_dip_aper
FROM #ejecucion_a_prelim_dip_aper
GROUP BY SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM,
         C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL
"""
work_conn.execute(text(sql_ejecucion_a_dip_aper))
ejecucion_a_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_a_dip_aper"), work_conn)
_log("ejecucion_a_dip_aper", ejecucion_a_dip_aper)


In [ ]:
# Analíticos variables: EJECUCION_B_DIP_aper (cruce con CR18_T_SCN_2, filtro C_CUENTA='32' y analíticos 04001000/04002000)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip_aper"))
sql_ejecucion_b_dip_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
       t1.MES,
       TRY_CAST(REPLACE(REPLACE(REPLACE(CONCAT('0', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', ''), CHAR(9), ''), CHAR(13), '') AS float) AS C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       CASE WHEN t4.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t4.NIVEL,
       CAST(t1.DEVENG * t3.DATO / 1000000.0 AS decimal(18,2)) AS Dato
INTO #ejecucion_b_dip_aper
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
     AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)}
  AND t1.MONEDA = '{Moneda}' AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_b_dip_aper))
res = work_conn.execute(text("DELETE FROM #ejecucion_b_dip_aper WHERE C_ENTIDAD IN (15004001)"))
_log("DELETE #ejecucion_b_dip_aper (C_ENTIDAD 15004001)", res.rowcount)
ejecucion_b_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_b_dip_aper"), work_conn)
_log("ejecucion_b_dip_aper", ejecucion_b_dip_aper)


In [ ]:
# EJECUCION_C_DIP_aper: cuentas de deuda/depreciación (05,13,24,33), excluye OBS de ajuste coyuntura/actividad y SCN='TC'
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_aper"))
sql_ejecucion_c_dip_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
       t1.MES,
       TRY_CAST(REPLACE(REPLACE(REPLACE(CONCAT('0', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', ''), CHAR(9), ''), CHAR(13), '') AS float) AS C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       CASE WHEN t4.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t4.NIVEL,
       CAST(t1.DEVENG * t3.DATO / 1000000.0 AS decimal(18,2)) AS Dato
INTO #ejecucion_c_dip_aper
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
     AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN #cr18_apertura_pacapr t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)}
  AND t1.MONEDA = '{Moneda}' AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_c_dip_aper))
res = work_conn.execute(text("DELETE FROM #ejecucion_c_dip_aper WHERE C_ENTIDAD IN (15004001)"))
_log("DELETE #ejecucion_c_dip_aper (C_ENTIDAD 15004001)", res.rowcount)
ejecucion_c_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_c_dip_aper"), work_conn)
_log("ejecucion_c_dip_aper", ejecucion_c_dip_aper)


In [ ]:
# Transacciones diferenciadas para coyuntura: EJECUCION_D_DIP_aper (OBS='CR18_difcoy', C_SCN toma el valor de C_ACTIVO)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip_aper"))
sql_ejecucion_d_dip_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
       t1.MES,
       TRY_CAST(REPLACE(REPLACE(REPLACE(CONCAT('0', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', ''), CHAR(9), ''), CHAR(13), '') AS float) AS C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_ACTIVO AS C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       CASE WHEN t4.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t4.NIVEL,
       CAST(t1.DEVENG * t3.DATO / 1000000.0 AS decimal(18,2)) AS Dato
INTO #ejecucion_d_dip_aper
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
     AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.ENTIDADES t5 ON t1.C_PARTIDA = t5.C_PARTIDA AND t1.C_CAPITULO = t5.C_CAPITULO AND t1.C_PROGRAMA = t5.C_PROGRAMA
LEFT JOIN #cr18_apertura_pacapr t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t2.OBS IN ('CR18_difcoy') AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.C_SCN NOT IN ('TC')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)}
  AND t1.MONEDA = '{Moneda}' AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_d_dip_aper))
ejecucion_d_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_d_dip_aper"), work_conn)
_log("ejecucion_d_dip_aper", ejecucion_d_dip_aper)


In [ ]:
# Transacciones diferenciadas por actividad: EJECUCION_F_DIP_aper (OBS='CR18_difact', C_ACT toma el valor de C_CUP)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip_aper"))
sql_ejecucion_f_dip_aper = f"""
SELECT t1.SUBSECTOR,
       t1.MONEDA,
       t1.AÑO,
       CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
       t1.MES,
       TRY_CAST(REPLACE(REPLACE(REPLACE(CONCAT('0', t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA), ' ', ''), CHAR(9), ''), CHAR(13), '') AS float) AS C_ENTIDAD,
       t1.C_PARTIDA,
       t1.C_CAPITULO,
       t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA,
       t1.C_CUENTA,
       t1.C_ITEM,
       t1.C_ASIGNACION,
       t1.C_ANALITICO,
       t1.N_ANALITICO,
       t2.C_SCN,
       t2.C_CAGENTE,
       t2.C_TRANSACCION,
       t2.T_TRANSACCION,
       t2.N_CUENTA,
       t2.C_ACTIVO,
       t2.C_CUP,
       t2.C_CUP AS C_ACT,
       t4.NIVEL,
       CAST(t1.DEVENG * t3.DATO / 1000000.0 AS decimal(18,2)) AS Dato
INTO #ejecucion_f_dip_aper
FROM GOBGENER.dbo.EJEC_DIPRES t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2 ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
     AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
     AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3 ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.ENTIDADES t5 ON t1.C_PARTIDA = t5.C_PARTIDA AND t1.C_CAPITULO = t5.C_CAPITULO AND t1.C_PROGRAMA = t5.C_PROGRAMA
LEFT JOIN #cr18_apertura_pacapr t4 ON t1.C_PARTIDA = t4.C_PARTIDA AND t1.C_CAPITULO = t4.C_CAPITULO AND t1.C_PROGRAMA = t4.C_PROGRAMA
WHERE t2.OBS IN ('CR18_difact') AND t2.C_SCN NOT IN ('TC') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.AÑO = {int(Year)} AND t1.MES = {int(Mes)}
  AND t1.MONEDA = '{Moneda}' AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejecucion_f_dip_aper))
ejecucion_f_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_f_dip_aper"), work_conn)
_log("ejecucion_f_dip_aper", ejecucion_f_dip_aper)


In [ ]:
# EJECUCION_G_DIP_aper: agregación desde EJEC_G_MIXT_aper_DIP (dejado por CodeTask-99VPAhpFklGwBLLO en memoria)
# ACTIVIDAD se descarta del resultado (DROP=ACTIVIDAD del SAS)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_aper_dip"))
ejec_g_mixt_aper_dip.to_sql("#ejec_g_mixt_aper_dip", work_conn, if_exists="replace", index=False)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip_aper"))
sql_ejecucion_g_dip_aper = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
       t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
       t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
       CASE WHEN t1.ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN t1.ACTIVIDAD = 'SSPP' THEN 'SSPP'
            WHEN t1.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
       t1.NIVEL,
       SUM(t1.dato) AS dato
INTO #ejecucion_g_dip_aper
FROM #ejec_g_mixt_aper_dip t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.MONEDA = '{Moneda}' AND t1.KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
         t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_g_dip_aper))
ejecucion_g_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_g_dip_aper"), work_conn)
_log("ejecucion_g_dip_aper", ejecucion_g_dip_aper)


In [ ]:
# UNE TABLAS de Salida total de ejecución CG coy de data DIPRES
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_prev"))
sql_ejecucion_cg_dip_aper_prev = """
SELECT * INTO #ejecucion_cg_dip_aper_prev FROM #ejecucion_a_dip_aper
UNION ALL SELECT * FROM #ejecucion_b_dip_aper
UNION ALL SELECT * FROM #ejecucion_c_dip_aper
UNION ALL SELECT * FROM #ejecucion_d_dip_aper
UNION ALL SELECT * FROM #ejecucion_g_dip_aper
UNION ALL SELECT * FROM #ejecucion_f_dip_aper
"""
work_conn.execute(text(sql_ejecucion_cg_dip_aper_prev))


In [ ]:
# DROP TABLE EJECUCION_A_PRELIM_DIP_APER,EJECUCION_A_DIP_aper, EJECUCION_B_DIP_aper, EJECUCION_C_DIP_aper,
# EJECUCION_D_DIP_aper, EJECUCION_G_DIP_aper, EJECUCION_F_DIP_aper (limpieza de #tmp intermedias)
for _t in ["#ejecucion_a_prelim_dip_aper", "#ejecucion_a_dip_aper", "#ejecucion_b_dip_aper", "#ejecucion_c_dip_aper",
           "#ejecucion_d_dip_aper", "#ejecucion_g_dip_aper", "#ejecucion_f_dip_aper"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


In [ ]:
# UPDATE EJECUCION_CG_DIP_aper_prev SET N_CUENTA='DIPRES'
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_aper_prev SET N_CUENTA = 'DIPRES'"))
_log("UPDATE #ejecucion_cg_dip_aper_prev (N_CUENTA=DIPRES)", res.rowcount)
ejecucion_cg_dip_aper_prev = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_aper_prev"), work_conn)
_log("ejecucion_cg_dip_aper_prev", ejecucion_cg_dip_aper_prev)


In [ ]:
# Incorporación de imputaciones permanentes: EJEC_CG_DIP_aper_IMPUT_PERM desde GOBGENER.IMPUT_GC_DIP_APER (dejada por CodeTask-iUHB7DjImHdoG82f)
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cg_dip_aper_imput_perm"))
sql_ejec_cg_dip_aper_imput_perm = f"""
SELECT SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA,
       C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO,
       C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP,
       CASE WHEN C_ACT IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST','AAPP') THEN 'AAPP'
            WHEN C_ACT = 'SSPP' THEN 'SSPP'
            WHEN C_ACT = 'EEPP' THEN 'EEPP' END AS C_ACT,
       NIVEL,
       Dato
INTO #ejec_cg_dip_aper_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER
WHERE AÑO = {int(Year)} AND C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND MONEDA = '{Moneda}' AND KEY IN ('{PACAPR1}','{PACAPR2}','{PACAPR3}','{PACAPR4}','{PACAPR5}','{PACAPR6}','{PACAPR7}')
"""
work_conn.execute(text(sql_ejec_cg_dip_aper_imput_perm))
ejec_cg_dip_aper_imput_perm = pd.read_sql(text("SELECT * FROM #ejec_cg_dip_aper_imput_perm"), work_conn)
_log("ejec_cg_dip_aper_imput_perm", ejec_cg_dip_aper_imput_perm)


In [ ]:
# EJECUCION_CG_DIP_aper: unión final de la salida coyuntural DIPRES con las imputaciones permanentes
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper"))
sql_ejecucion_cg_dip_aper = """
SELECT * INTO #ejecucion_cg_dip_aper FROM #ejecucion_cg_dip_aper_prev
UNION ALL SELECT * FROM #ejec_cg_dip_aper_imput_perm
"""
work_conn.execute(text(sql_ejecucion_cg_dip_aper))
ejecucion_cg_dip_aper = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_aper"), work_conn)
_log("ejecucion_cg_dip_aper", ejecucion_cg_dip_aper)


In [ ]:
# DROP TABLE EJEC_CG_DIP_aper_IMPUT_PERM; DROP TABLE CR18_APERTURA_PACAPR;
# (EJEC_CG_DIP_APER_IMPUT_PERM se mantiene como #tmp para nodos posteriores del flujo: no se dropea acá pese a que el SAS lo hacía,
# porque otras dependencias del notebook la consumen por su nombre real ejec_cg_dip_aper_imput_perm)
work_conn.execute(text("DROP TABLE IF EXISTS #cr18_apertura_pacapr"))
